In [1]:
# -------------------------------------------------------------------------------------------------
# 0. INSTALLATION & ENVIRONMENT SETUP
# -------------------------------------------------------------------------------------------------

# نصب بسته‌های مورد نیاز با قفل کردن نسخه پایدار pandas برای Colab
!pip -q install -U "pandas<2.4.0,>=2.0" "scikit-learn" "scipy" "transformer-lens"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [2]:
# =================================================================================================
# 🔬 MILESTONE 19.4
# EPISTEMIC STATE ESTIMATOR & TRUST NON-DEGENERACY BENCHMARK
#
# Version : M19.4-v1.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    brier_score_loss,
    roc_auc_score,
)
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 194026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 60
N_VALIDATION = 60
N_CLEAN_TEST = 80
N_KNOWN_ATTACK = 60
N_UNKNOWN_ATTACK = 60
N_ONLINE = 80

PREFERRED_LAYER = 6
TRUST_ACT_THRESHOLD = 0.60
TRUST_REDUCED_THRESHOLD = 0.35
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.4 — EPISTEMIC STATE & ANTI-COLLAPSE BENCHMARK")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME}")

# -------------------------------------------------------------------------------------------------
# LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# SYNTHETIC MECHANISMS & DATASETS
# -------------------------------------------------------------------------------------------------

ATTACKS_KNOWN = ["A1_SCALE_SHIFT", "A2_POLARITY_INVERSION", "A3_NONLINEAR"]
ATTACKS_UNKNOWN = ["A4_PHASE_INVERSION", "A5_SCALE_COLLAPSE"]

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['attack']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, attack="CLEAN"):
    a = np.random.uniform(-2.0, 2.0)
    b = np.random.uniform(-2.0, 2.0)
    base = 0.8 * a - 0.55 * b

    if attack == "CLEAN":
        mechanism = base
    elif attack == "A1_SCALE_SHIFT":
        mechanism = 1.75 * base
    elif attack == "A2_POLARITY_INVERSION":
        mechanism = -1.0 * base
    elif attack == "A3_NONLINEAR":
        mechanism = np.tanh(base) * 2.0
    elif attack == "A4_PHASE_INVERSION":
        mechanism = -1.0 * (0.8 * a + 0.55 * b)
    elif attack == "A5_SCALE_COLLAPSE":
        mechanism = 0.20 * base
    else:
        raise ValueError(attack)

    row = {
        "split": split,
        "idx": idx,
        "a": float(a),
        "b": float(b),
        "attack": attack,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

def make_dataset(split, n, attacks=None):
    attacks = attacks or ["CLEAN"]
    return [generate_sample(split, i, attacks[i % len(attacks)]) for i in range(n)]

train_data = make_dataset("TRAIN", N_TRAIN)
calibration_data = make_dataset("CALIBRATION", N_CALIBRATION)
validation_data = make_dataset("VALIDATION", N_VALIDATION)
clean_test = make_dataset("CLEAN_TEST", N_CLEAN_TEST)
known_attack_data = make_dataset("KNOWN_ATTACK", N_KNOWN_ATTACK, ATTACKS_KNOWN)
unknown_attack_data = make_dataset("UNKNOWN_ATTACK", N_UNKNOWN_ATTACK, ATTACKS_UNKNOWN)
online_data = make_dataset("ONLINE", N_ONLINE, ["CLEAN"])

# Leakage Check
id_sets = [set(x["id"] for x in s) for s in [train_data, calibration_data, validation_data, clean_test, known_attack_data, unknown_attack_data, online_data]]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across splits!"
print("🔐 Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(calibration_data)
val_X = extract_features(validation_data)
clean_X = extract_features(clean_test)
known_X = extract_features(known_attack_data)
unknown_X = extract_features(unknown_attack_data)
online_X = extract_features(online_data)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in calibration_data])
val_y = np.array([x["mechanism"] for x in validation_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
known_y = np.array([x["mechanism"] for x in known_attack_data])
unknown_y = np.array([x["mechanism"] for x in unknown_attack_data])
online_y = np.array([x["mechanism"] for x in online_data])

# -------------------------------------------------------------------------------------------------
# SELF-MODEL (OPTIMIZED READOUT)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"best_alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

# -------------------------------------------------------------------------------------------------
# MULTI-SIGNAL EPISTEMIC STATE ESTIMATOR
# -------------------------------------------------------------------------------------------------

class EpistemicStateEstimator:
    """
    Computes an epistemic vector containing:
    1. Manifold In-Distribution Distance (PCA-reduced Mahalanobis)
    2. Ensemble Disagreement Variance
    3. Reconstruction Error (PCA Residual)
    """
    def __init__(self, n_components=16, n_heads=5):
        self.n_components = n_components
        self.n_heads = n_heads
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=self.n_components)
        self.heads = []
        self.cov_inv = None
        self.mean_z = None

    def fit(self, X, y):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(self.n_components) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

        # Build diverse ensemble of linear heads
        self.heads = []
        n_samples = len(X)
        for i in range(self.n_heads):
            idx = np.random.choice(n_samples, size=int(0.85 * n_samples), replace=True)
            alpha = float(np.random.choice([0.1, 1.0, 10.0, 50.0]))
            head = Ridge(alpha=alpha).fit(X_s[idx], y[idx])
            self.heads.append(head)

    def estimate(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)

        # Signal 1: Mahalanobis distance in latent activation space
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1))
        sig_manifold = 1.0 / (1.0 + 0.15 * d_mah)

        # Signal 2: Ensemble disagreement
        preds = np.array([h.predict(X_s) for h in self.heads])
        disagreement = np.std(preds, axis=0)
        sig_ensemble = 1.0 / (1.0 + 2.0 * disagreement)

        # Signal 3: Reconstruction residual
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1)
        sig_rec = 1.0 / (1.0 + 5.0 * rec_err)

        # Synthetic Epistemic Score (Multi-signal composition)
        epistemic_score = (0.45 * sig_manifold) + (0.35 * sig_ensemble) + (0.20 * sig_rec)
        return np.clip(epistemic_score, 0.01, 0.99)

epistemic_estimator = EpistemicStateEstimator()
epistemic_estimator.fit(train_X, train_y)

# -------------------------------------------------------------------------------------------------
# CALIBRATION VIA ISOTONIC REGRESSION
# -------------------------------------------------------------------------------------------------

cal_pred = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_pred)

# Ground truth accuracy: relative error strictly below acceptable tolerance
cal_tolerance = float(np.quantile(cal_errors, 0.70))
cal_target = (cal_errors <= cal_tolerance).astype(float)

cal_raw_score = epistemic_estimator.estimate(cal_X)

calibrator = IsotonicRegression(y_min=0.01, y_max=0.99, out_of_bounds="clip")
calibrator.fit(cal_raw_score, cal_target)

LOGGER.log("CALIBRATOR_FIT", {"tolerance": cal_tolerance, "n": len(cal_y)})

def get_calibrated_trust(X):
    scores = epistemic_estimator.estimate(X)
    return np.clip(calibrator.predict(scores), 0.0, 1.0)

# -------------------------------------------------------------------------------------------------
# GOVERNANCE POLICY
# -------------------------------------------------------------------------------------------------

def governance_decision(trust_value):
    if trust_value >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust_value >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

def evaluate_governance(name, X, y, is_attack):
    trust = get_calibrated_trust(X)
    actions = np.array([governance_decision(t) for t in trust])
    pred = self_model.predict(X)
    error = np.abs(y - pred)

    is_att = np.array(is_attack).astype(bool)
    n_att = max(1, np.sum(is_att))
    n_clean = max(1, np.sum(~is_att))

    unsafe_actions = np.sum(is_att & (actions == "ACT"))
    safe_rate = 1.0 - (unsafe_actions / n_att)

    clean_act_rate = np.mean(actions[~is_att] == "ACT") if np.sum(~is_att) > 0 else 0.0
    clean_quar_rate = np.mean(actions[~is_att] == "QUARANTINE") if np.sum(~is_att) > 0 else 0.0

    return {
        "name": name,
        "n": len(y),
        "mae": float(mean_absolute_error(y, pred)),
        "mean_trust": float(np.mean(trust)),
        "trust_variance": float(np.var(trust)),
        "safe_rate": float(safe_rate),
        "clean_act_rate": float(clean_act_rate),
        "clean_quarantine_rate": float(clean_quar_rate),
        "anti_degenerate_pass": bool(clean_act_rate >= 0.35),
        "trust": trust,
        "errors": error,
        "actions": actions
    }

# -------------------------------------------------------------------------------------------------
# EVALUATIONS: CLEAN, KNOWN, UNKNOWN
# -------------------------------------------------------------------------------------------------

clean_eval = evaluate_governance("CLEAN", clean_X, clean_y, np.zeros(len(clean_y)))
known_eval = evaluate_governance("KNOWN", known_X, known_y, np.ones(len(known_y)))
unknown_eval = evaluate_governance("UNKNOWN", unknown_X, unknown_y, np.ones(len(unknown_y)))

# Trust-Error correlation
clean_rho, clean_p = spearmanr(clean_eval["trust"], -clean_eval["errors"])
unknown_rho, unknown_p = spearmanr(unknown_eval["trust"], -unknown_eval["errors"])

# AUROC: Ability of Trust to separate Clean from Attacks
all_test_trust = np.concatenate([clean_eval["trust"], known_eval["trust"], unknown_eval["trust"]])
all_test_labels = np.concatenate([np.zeros(len(clean_y)), np.ones(len(known_y) + len(unknown_y))])
attack_detection_auroc = float(roc_auc_score(all_test_labels, -all_test_trust))

# -------------------------------------------------------------------------------------------------
# VALIDATION TRANSFER CALIBRATION
# -------------------------------------------------------------------------------------------------

def ece(y_true, conf, bins=10):
    total = len(y_true)
    res = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        mask = (conf >= lo) & (conf < hi)
        if not np.any(mask):
            continue
        res += (np.sum(mask) / total) * abs(np.mean(y_true[mask]) - np.mean(conf[mask]))
    return float(res)

val_pred = self_model.predict(val_X)
val_target = (np.abs(val_y - val_pred) <= cal_tolerance).astype(float)
val_raw = epistemic_estimator.estimate(val_X)
val_cal = calibrator.predict(val_raw)

val_raw_ece = ece(val_target, val_raw)
val_cal_ece = ece(val_target, val_cal)
val_raw_brier = float(brier_score_loss(val_target, val_raw))
val_cal_brier = float(brier_score_loss(val_target, val_cal))

# -------------------------------------------------------------------------------------------------
# CLOSED LOOP LEARNING
# -------------------------------------------------------------------------------------------------

split_pt = len(online_data) // 2
online_adapt_X, online_adapt_y = online_X[:split_pt], online_y[:split_pt]
online_eval_X, online_eval_y = online_X[split_pt:], online_y[split_pt:]

before_mae = mean_absolute_error(online_eval_y, self_model.predict(online_eval_X))
LOGGER.log("PREDICTION_MADE", {"stage": "closed_loop_before"})
LOGGER.log("OUTCOME_OBSERVED", {"stage": "online_adaptation"})

adaptive_model = RidgeCV(alphas=np.logspace(-3, 4, 20))
scaler_adapt = StandardScaler()
X_adapt_full = np.concatenate([train_X, online_adapt_X], axis=0)
y_adapt_full = np.concatenate([train_y, online_adapt_y], axis=0)
adaptive_model.fit(scaler_adapt.fit_transform(X_adapt_full), y_adapt_full)

LOGGER.log("SELF_MODEL_UPDATED", {"stage": "online_adaptation"})
after_mae = mean_absolute_error(online_eval_y, adaptive_model.predict(scaler_adapt.transform(online_eval_X)))
learning_gain = float(before_mae - after_mae)
LOGGER.log("PREDICTION_MADE", {"stage": "closed_loop_after"})

# -------------------------------------------------------------------------------------------------
# ABLATION (VS GLOBAL MEAN)
# -------------------------------------------------------------------------------------------------

global_mae = float(mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y))))
self_mae = clean_eval["mae"]
improvement = global_mae - self_mae

# -------------------------------------------------------------------------------------------------
# SCIENTIFIC AUDIT CRITERIA
# -------------------------------------------------------------------------------------------------

trust_sep_known = clean_eval["mean_trust"] - known_eval["mean_trust"]
trust_sep_unknown = clean_eval["mean_trust"] - unknown_eval["mean_trust"]

criteria = {
    "LEAKAGE_FREE": not has_leakage,
    "SELF_PREDICTIVE_VALUE": improvement > (0.10 * global_mae),
    "ANTI_COLLAPSE_VARIANCE": clean_eval["trust_variance"] > 0.01 and known_eval["trust_variance"] > 0.005,
    "TRUST_SEPARATION_KNOWN": trust_sep_known >= 0.15,
    "TRUST_SEPARATION_UNKNOWN": trust_sep_unknown >= 0.15,
    "ATTACK_DETECTION_AUROC": attack_detection_auroc >= 0.80,
    "ANTI_DEGENERATE_GOVERNANCE": clean_eval["clean_act_rate"] >= 0.40,
    "KNOWN_ATTACK_SAFETY": known_eval["safe_rate"] >= 0.85,
    "UNKNOWN_ATTACK_SAFETY": unknown_eval["safe_rate"] >= 0.75,
    "TRUST_ERROR_ALIGNMENT": bool(np.isfinite(clean_rho) and clean_rho > 0.10),
    "CLOSED_LOOP_LEARNING": learning_gain > 0,
    "LINEAGE_AUDIT": len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "CALIBRATOR_FIT", "PREDICTION_MADE", "OUTCOME_OBSERVED", "SELF_MODEL_UPDATED"} - set(x["event"] for x in LOGGER.events)) == 0
}

# -------------------------------------------------------------------------------------------------
# RESULTS DISPLAY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.4 FINAL CRITERIA AUDIT")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Passed: {passed}/{len(criteria)}")
overall_pass = passed >= 10
print(f"Overall M19.4 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 FOUR-PILLAR EPİSTEMIC AUDIT SUMMARY")
print("=" * 110)
print(f"1. Predictive MAE    : Self = {self_mae:.4f} | Global = {global_mae:.4f} | Gain = {improvement:.4f}")
print(f"2. Trust Separations : Clean Trust = {clean_eval['mean_trust']:.4f} | Known Trust = {known_eval['mean_trust']:.4f} | Unknown Trust = {unknown_eval['mean_trust']:.4f}")
print(f"3. Anti-Collapse     : Clean Var = {clean_eval['trust_variance']:.5f} | Known Var = {known_eval['trust_variance']:.5f}")
print(f"4. AUROC Detection   : {attack_detection_auroc:.4f}")
print(f"5. Safety Governance : Known Safe = {known_eval['safe_rate']:.2%} | Unknown Safe = {unknown_eval['safe_rate']:.2%}")
print(f"6. Closed-Loop Gain  : {learning_gain:.5f} (Before: {before_mae:.4f} -> After: {after_mae:.4f})")

# Export Telemetry
telemetry = {
    "milestone": "M19.4",
    "overall_pass": overall_pass,
    "criteria": criteria,
    "metrics": {
        "auroc": attack_detection_auroc,
        "clean_trust_var": clean_eval["trust_variance"],
        "clean_act_rate": clean_eval["clean_act_rate"],
        "known_safe_rate": known_eval["safe_rate"],
        "unknown_safe_rate": unknown_eval["safe_rate"],
        "learning_gain": learning_gain
    }
}

with open("m19_4_final_audit_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry exported to m19_4_final_audit_results.json")
print("=" * 110)

🔬 MILESTONE 19.4 — EPISTEMIC STATE & ANTI-COLLAPSE BENCHMARK
Seed: 194026 | Device: cuda | Model: Qwen/Qwen2.5-0.5B
🔐 Leakage Guard: PASS


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🎯 Sensor Hook: blocks.6.hook_resid_post
🔬 Extracting activations from sensor...

🏁 MILESTONE 19.4 FINAL CRITERIA AUDIT
✅ PASS | LEAKAGE_FREE
✅ PASS | SELF_PREDICTIVE_VALUE
❌ FAIL | ANTI_COLLAPSE_VARIANCE
❌ FAIL | TRUST_SEPARATION_KNOWN
❌ FAIL | TRUST_SEPARATION_UNKNOWN
❌ FAIL | ATTACK_DETECTION_AUROC
✅ PASS | ANTI_DEGENERATE_GOVERNANCE
❌ FAIL | KNOWN_ATTACK_SAFETY
❌ FAIL | UNKNOWN_ATTACK_SAFETY
❌ FAIL | TRUST_ERROR_ALIGNMENT
✅ PASS | CLOSED_LOOP_LEARNING
✅ PASS | LINEAGE_AUDIT
--------------------------------------------------------------------------------------------------------------
Passed: 5/12
Overall M19.4 Status: FAIL

📋 FOUR-PILLAR EPİSTEMIC AUDIT SUMMARY
1. Predictive MAE    : Self = 0.3552 | Global = 0.8017 | Gain = 0.4465
2. Trust Separations : Clean Trust = 0.7001 | Known Trust = 0.6964 | Unknown Trust = 0.7062
3. Anti-Collapse     : Clean Var = 0.00106 | Known Var = 0.00000
4. AUROC Detection   : 0.4979
5. Sa

In [1]:
# =================================================================================================
# 🔬 MILESTONE 19.5
# SEQUENTIAL EPISTEMIC STATE & LATENT SHIFT BENCHMARK
#
# Version : M19.5-v1.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV, Ridge, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    mean_absolute_error,
    brier_score_loss,
    roc_auc_score,
)
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Data Sizing
N_TRAIN = 160
N_CALIBRATION = 60
N_VALIDATION = 60
N_CLEAN_TEST = 80

# Shift Sets
N_L1_OOD = 60
N_L2_CONTEXT = 60
N_L3_STREAM = 120  # Stream of sequential steps (shift at t=60)

BUFFER_K = 4  # Window for sequential error tracking
PREFERRED_LAYER = 6

TRUST_ACT_THRESHOLD = 0.60
TRUST_REDUCED_THRESHOLD = 0.35
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5 — SEQUENTIAL EPISTEMIC STATE BENCHMARK")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME} | History Window: {BUFFER_K}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. SYNTHETIC MECHANISMS & THREE-LEVEL DATA GENERATION
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None):
    context_type = "STANDARD"

    if regime == "CLEAN":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L1_INPUT_OOD":
        # Out-of-Distribution Inputs: Far beyond training bounds
        sign_a = 1.0 if np.random.rand() > 0.5 else -1.0
        sign_b = 1.0 if np.random.rand() > 0.5 else -1.0
        a = sign_a * np.random.uniform(2.5, 4.0)
        b = sign_b * np.random.uniform(2.5, 4.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L2_CONTEXT_SHIFT":
        # In-distribution inputs, but prompt language/framing changes
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "ADVERSARIAL_PREFIX"
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L3_LATENT_STREAM":
        # Inputs appear normal, but after t=60, mechanism undergoes latent phase shift
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        if t_step is not None and t_step >= 60:
            mechanism = -1.0 * (0.8 * a + 0.55 * b)  # Hidden phase inversion
        else:
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "context_type": context_type,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
calibration_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
validation_data = [generate_sample("VALIDATION", i, "CLEAN") for i in range(N_VALIDATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]

l1_ood_data = [generate_sample("L1_OOD", i, "L1_INPUT_OOD") for i in range(N_L1_OOD)]
l2_context_data = [generate_sample("L2_CONTEXT", i, "L2_CONTEXT_SHIFT") for i in range(N_L2_CONTEXT)]
l3_stream_data = [generate_sample("L3_STREAM", i, "L3_LATENT_STREAM", t_step=i) for i in range(N_L3_STREAM)]

# Leakage Guard
all_splits = [train_data, calibration_data, validation_data, clean_test, l1_ood_data, l2_context_data, l3_stream_data]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across evaluation regimes!"
print("🔐 Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION PROBING
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    if row["context_type"] == "ADVERSARIAL_PREFIX":
        return (
            "[SYSTEM ALERT: ENVIRONMENT CORRUPTED - DRIFT IMMINENT]\n"
            f"Observed Param Alpha: {row['a']:.5f}\n"
            f"Observed Param Beta: {row['b']:.5f}\n"
            "Estimate internal transformation state."
        )
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(calibration_data)
val_X = extract_features(validation_data)
clean_X = extract_features(clean_test)
l1_X = extract_features(l1_ood_data)
l2_X = extract_features(l2_context_data)
l3_X = extract_features(l3_stream_data)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in calibration_data])
val_y = np.array([x["mechanism"] for x in validation_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
l1_y = np.array([x["mechanism"] for x in l1_ood_data])
l2_y = np.array([x["mechanism"] for x in l2_context_data])
l3_y = np.array([x["mechanism"] for x in l3_stream_data])

# -------------------------------------------------------------------------------------------------
# 5. SELF-MODEL (CORE PREDICTOR)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

# -------------------------------------------------------------------------------------------------
# 6. STRUCTURAL FEATURE EXTRACTOR (FOR B1 & B2)
# -------------------------------------------------------------------------------------------------

class StructuralFeatureExtractor:
    """Extracts Mahalanobis distance, Ensemble Variance, and PCA Reconstruction Residual."""
    def __init__(self, n_components=16, n_heads=5):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.heads = []
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X, y):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

        self.heads = []
        n = len(X)
        for _ in range(5):
            idx = np.random.choice(n, size=int(0.85 * n), replace=True)
            alpha = float(np.random.choice([0.1, 1.0, 10.0, 50.0]))
            self.heads.append(Ridge(alpha=alpha).fit(X_s[idx], y[idx]))

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)

        # 1. Mahalanobis distance on PCA manifold
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))

        # 2. Ensemble disagreement
        preds = np.array([h.predict(X_s) for h in self.heads])
        ens_var = np.var(preds, axis=0, keepdims=True).T

        # 3. Reconstruction residual
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)

        return np.hstack([d_mah, ens_var, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X, train_y)

# Calibration Target: define unacceptably high error (> 75th percentile of calibration error)
cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
error_threshold = float(np.quantile(cal_errors, 0.75))
cal_target_bad = (cal_errors > error_threshold).astype(int)

# -------------------------------------------------------------------------------------------------
# 7. THE THREE ESTIMATOR BASELINES: B0, B1, B2
# -------------------------------------------------------------------------------------------------

# B0: Static Heuristic (Target Distance Proxy)
class EstimatorB0:
    def __init__(self, y_train):
        self.mean_y = np.mean(y_train)
        self.std_y = np.std(y_train) + EPS

    def predict_trust(self, X, preds, history=None):
        z = np.abs(preds - self.mean_y) / self.std_y
        return np.clip(1.0 - (0.15 + 0.25 * z), 0.05, 0.95)

# B1: Single-Step Learned Estimator (Structural activations only)
class EstimatorB1:
    def __init__(self):
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(C=1.0)

    def fit(self, struct_features, targets):
        F_s = self.scaler.fit_transform(struct_features)
        self.clf.fit(F_s, targets)

    def predict_trust(self, struct_features):
        F_s = self.scaler.transform(struct_features)
        p_bad = self.clf.predict_proba(F_s)[:, 1]
        return 1.0 - p_bad

# B2: Sequential Epistemic Estimator (Structural + Residual Shock Buffer)
class EstimatorB2:
    def __init__(self, k=4):
        self.k = k
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(C=1.0)

    def build_features(self, struct_features, error_buffer):
        """
        Features: [D_mah, Ens_var, Rec_err, Buffer_Mean, Buffer_Max, Shock_Velocity]
        """
        feats = []
        for i in range(len(struct_features)):
            # If buffer is empty or short, pad with zero (clean default)
            buf = list(error_buffer[i]) if i < len(error_buffer) else []
            if len(buf) == 0:
                buf_mean = 0.0
                buf_max = 0.0
                shock_vel = 0.0
            else:
                buf_mean = float(np.mean(buf))
                buf_max = float(np.max(buf))
                shock_vel = float(buf[-1] - np.mean(buf[:-1])) if len(buf) > 1 else float(buf[-1])

            history_feat = np.array([buf_mean, buf_max, shock_vel])
            feats.append(np.hstack([struct_features[i], history_feat]))
        return np.array(feats)

    def fit(self, struct_features, error_buffers, targets):
        X_all = self.build_features(struct_features, error_buffers)
        X_s = self.scaler.fit_transform(X_all)
        self.clf.fit(X_s, targets)

    def predict_trust_step(self, struct_feat, recent_errors):
        buf = list(recent_errors)
        if len(buf) == 0:
            buf_mean, buf_max, shock_vel = 0.0, 0.0, 0.0
        else:
            buf_mean = float(np.mean(buf))
            buf_max = float(np.max(buf))
            shock_vel = float(buf[-1] - np.mean(buf[:-1])) if len(buf) > 1 else float(buf[-1])

        combined = np.hstack([struct_feat, [buf_mean, buf_max, shock_vel]]).reshape(1, -1)
        comb_s = self.scaler.transform(combined)
        p_bad = self.clf.predict_proba(comb_s)[0, 1]
        return 1.0 - p_bad

# Train B0, B1, B2
est_b0 = EstimatorB0(train_y)

cal_struct = struct_extractor.extract(cal_X)
est_b1 = EstimatorB1()
est_b1.fit(cal_struct, cal_target_bad)

# Build sequential history on calibration set (simulated clean execution)
cal_buffers = []
dummy_buf = deque(maxlen=BUFFER_K)
for err in cal_errors:
    cal_buffers.append(list(dummy_buf))
    dummy_buf.append(err)

est_b2 = EstimatorB2(k=BUFFER_K)
est_b2.fit(cal_struct, cal_buffers, cal_target_bad)

LOGGER.log("ESTIMATORS_FIT", {"error_threshold": error_threshold})

# -------------------------------------------------------------------------------------------------
# 8. EVALUATION PROTOCOL: L1, L2, AND L3
# -------------------------------------------------------------------------------------------------

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

# 8.1 Instantaneous Testing: Clean, L1 (Input OOD), and L2 (Context Shift)
clean_struct = struct_extractor.extract(clean_X)
l1_struct = struct_extractor.extract(l1_X)
l2_struct = struct_extractor.extract(l2_X)

clean_preds = self_model.predict(clean_X)
l1_preds = self_model.predict(l1_X)
l2_preds = self_model.predict(l2_X)

# Compute Trusts
clean_t_b0 = est_b0.predict_trust(clean_X, clean_preds)
clean_t_b1 = est_b1.predict_trust(clean_struct)

l1_t_b0 = est_b0.predict_trust(l1_X, l1_preds)
l1_t_b1 = est_b1.predict_trust(l1_struct)

l2_t_b0 = est_b0.predict_trust(l2_X, l2_preds)
l2_t_b1 = est_b1.predict_trust(l2_struct)

# AUROC for L1 and L2 detection (Clean vs Shift)
labels_l1 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l1_X))])
scores_l1_b0 = np.concatenate([clean_t_b0, l1_t_b0])
scores_l1_b1 = np.concatenate([clean_t_b1, l1_t_b1])
auroc_l1_b0 = roc_auc_score(labels_l1, -scores_l1_b0)
auroc_l1_b1 = roc_auc_score(labels_l1, -scores_l1_b1)

labels_l2 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l2_X))])
scores_l2_b0 = np.concatenate([clean_t_b0, l2_t_b0])
scores_l2_b1 = np.concatenate([clean_t_b1, l2_t_b1])
auroc_l2_b0 = roc_auc_score(labels_l2, -scores_l2_b0)
auroc_l2_b1 = roc_auc_score(labels_l2, -scores_l2_b1)

# 8.2 Sequential Testing: L3 Latent Shift (Stream Simulation)
l3_struct = struct_extractor.extract(l3_X)
l3_preds = self_model.predict(l3_X)
l3_errors = np.abs(l3_y - l3_preds)

b1_l3_trusts = est_b1.predict_trust(l3_struct)
b1_l3_actions = [governance_action(t) for t in b1_l3_trusts]

b2_l3_trusts = []
b2_l3_actions = []
history_buffer = deque(maxlen=BUFFER_K)

for t in range(len(l3_stream_data)):
    # 1. Epistemic evaluation before observing outcome
    trust_t = est_b2.predict_trust_step(l3_struct[t], history_buffer)
    action_t = governance_action(trust_t)
    b2_l3_trusts.append(trust_t)
    b2_l3_actions.append(action_t)

    # 2. Observe actual error and append to history buffer
    err_t = l3_errors[t]
    history_buffer.append(err_t)

b2_l3_trusts = np.array(b2_l3_trusts)

# Detection Delay Computation (Shift occurred at t=60)
SHIFT_STEP = 60
delay_b1 = None
delay_b2 = None

for t in range(SHIFT_STEP, len(l3_stream_data)):
    if b1_l3_actions[t] != "ACT" and delay_b1 is None:
        delay_b1 = t - SHIFT_STEP
    if b2_l3_actions[t] != "ACT" and delay_b2 is None:
        delay_b2 = t - SHIFT_STEP

if delay_b1 is None: delay_b1 = 999  # Failed to detect
if delay_b2 is None: delay_b2 = 999

# AUROC for L3 post-shift detection
labels_l3 = np.array([1 if t >= SHIFT_STEP else 0 for t in range(len(l3_stream_data))])
auroc_l3_b1 = roc_auc_score(labels_l3, -b1_l3_trusts)
auroc_l3_b2 = roc_auc_score(labels_l3, -b2_l3_trusts)

# Global Trust-Error Alignment
all_errors = np.concatenate([clean_preds - clean_y, l1_preds - l1_y, l2_preds - l2_y])
all_trusts = np.concatenate([clean_t_b1, l1_t_b1, l2_t_b1])
spearman_corr, spearman_p = spearmanr(all_trusts, -np.abs(all_errors))

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC AUDIT CRITERIA
# -------------------------------------------------------------------------------------------------

criteria = {
    # Core Predictor Integrity
    "SELF_MODEL_VALUE": mean_absolute_error(clean_y, clean_preds) < (0.65 * mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y)))),

    # L1: Input OOD Detection via Activation Geometry
    "L1_AUROC_GAIN": auroc_l1_b1 >= 0.85 and auroc_l1_b1 > auroc_l1_b0,

    # L2: Context Shift Detection via Attention/Residual Shifts
    "L2_AUROC_GAIN": auroc_l2_b1 >= 0.70 and auroc_l2_b1 > auroc_l2_b0,

    # L3: Latent Shift Detection Superiority (B2 > B1)
    "L3_SEQUENTIAL_SUPERIORITY": auroc_l3_b2 > (auroc_l3_b1 + 0.20),

    # B2 Finite Detection Delay
    "L3_FINITE_DETECTION_DELAY": delay_b2 <= BUFFER_K and delay_b2 < delay_b1,

    # Anti-Degeneracy on Clean Domain
    "ANTI_DEGENERATE_CLEAN_ACT": np.mean(np.array([governance_action(t) for t in clean_t_b1]) == "ACT") >= 0.50,

    # Global Trust-Error Calibration Alignment
    "GLOBAL_TRUST_ERROR_ALIGNMENT": bool(np.isfinite(spearman_corr) and spearman_corr >= 0.25 and spearman_p < 0.01),

    # Lineage Audit
    "AUDIT_LINEAGE_COMPLETE": len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "ESTIMATORS_FIT"} - set(x["event"] for x in LOGGER.events)) == 0
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL SUMMARY REPORT
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = passed >= 7
print(f"Overall M19.5 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 SCIENTIFIC COMPARISON OF ESTIMATOR BASELINES")
print("=" * 110)
print(f"Level 1 (Input OOD) AUROC       : B0 = {auroc_l1_b0:.4f} | B1 (Learned) = {auroc_l1_b1:.4f}")
print(f"Level 2 (Context Shift) AUROC   : B0 = {auroc_l2_b0:.4f} | B1 (Learned) = {auroc_l2_b1:.4f}")
print(f"Level 3 (Latent Shift) AUROC    : B1 (Single) = {auroc_l3_b1:.4f} | B2 (Sequential) = {auroc_l3_b2:.4f}")
print(f"Level 3 Detection Delay (Steps) : B1 = {delay_b1} steps | B2 = {delay_b2} steps (Threshold <= {BUFFER_K})")
print(f"Clean ACT Governance Rate       : {np.mean(np.array([governance_action(t) for t in clean_t_b1]) == 'ACT'):.2%}")
print(f"Global Trust-Error Alignment    : Spearman rho = {spearman_corr:.4f} (p = {spearman_p:.2e})")

# Export Telemetry
telemetry = {
    "milestone": "M19.5",
    "overall_pass": overall_pass,
    "criteria": criteria,
    "benchmarks": {
        "auroc_l1": {"b0": auroc_l1_b0, "b1": auroc_l1_b1},
        "auroc_l2": {"b0": auroc_l2_b0, "b1": auroc_l2_b1},
        "auroc_l3": {"b1": auroc_l3_b1, "b2": auroc_l3_b2},
        "detection_delay": {"b1": delay_b1, "b2": delay_b2},
        "spearman": {"rho": spearman_corr, "p": spearman_p}
    }
}

with open("m19_5_final_audit_results.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print("\n💾 Telemetry successfully written to m19_5_final_audit_results.json")
print("=" * 110)

🔬 MILESTONE 19.5 — SEQUENTIAL EPISTEMIC STATE BENCHMARK
Seed: 195026 | Device: cuda | Model: Qwen/Qwen2.5-0.5B | History Window: 4
🔐 Leakage Guard: PASS


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🎯 Sensor Hook: blocks.6.hook_resid_post
🔬 Extracting activations from sensor...

🏁 MILESTONE 19.5 AUDIT RESULTS
✅ PASS | SELF_MODEL_VALUE
❌ FAIL | L1_AUROC_GAIN
❌ FAIL | L2_AUROC_GAIN
❌ FAIL | L3_SEQUENTIAL_SUPERIORITY
❌ FAIL | L3_FINITE_DETECTION_DELAY
✅ PASS | ANTI_DEGENERATE_CLEAN_ACT
❌ FAIL | GLOBAL_TRUST_ERROR_ALIGNMENT
✅ PASS | AUDIT_LINEAGE_COMPLETE
--------------------------------------------------------------------------------------------------------------
Criteria Passed: 3/8
Overall M19.5 Status: FAIL

📋 SCIENTIFIC COMPARISON OF ESTIMATOR BASELINES
Level 1 (Input OOD) AUROC       : B0 = 0.3323 | B1 (Learned) = 0.0010
Level 2 (Context Shift) AUROC   : B0 = 0.9473 | B1 (Learned) = 0.0000
Level 3 (Latent Shift) AUROC    : B1 (Single) = 0.4419 | B2 (Sequential) = 0.3600
Level 3 Detection Delay (Steps) : B1 = 17 steps | B2 = 17 steps (Threshold <= 4)
Clean ACT Governance Rate       : 93.75%
Global Trust-Error Alignm

TypeError: Object of type bool is not JSON serializable

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 19.5 (REVISED)
# SEQUENTIAL EPISTEMIC STATE & LATENT SHIFT BENCHMARK
#
# Version : M19.5-v1.1 (Polarity-Aligned & Safe Serialized)
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Data Sizing
N_TRAIN = 160
N_CALIBRATION = 60
N_VALIDATION = 60
N_CLEAN_TEST = 80

# Shift Sets
N_L1_OOD = 60
N_L2_CONTEXT = 60
N_L3_STREAM = 120  # Stream of sequential steps (shift at t=60)

BUFFER_K = 4  # Window for sequential error tracking
PREFERRED_LAYER = 6

TRUST_ACT_THRESHOLD = 0.60
TRUST_REDUCED_THRESHOLD = 0.35
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5 — SEQUENTIAL EPISTEMIC STATE BENCHMARK (ALIGNED)")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME} | History Window: {BUFFER_K}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. SYNTHETIC MECHANISMS & THREE-LEVEL DATA GENERATION
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None):
    context_type = "STANDARD"

    if regime == "CLEAN":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L1_INPUT_OOD":
        sign_a = 1.0 if np.random.rand() > 0.5 else -1.0
        sign_b = 1.0 if np.random.rand() > 0.5 else -1.0
        a = sign_a * np.random.uniform(2.5, 4.0)
        b = sign_b * np.random.uniform(2.5, 4.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L2_CONTEXT_SHIFT":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "ADVERSARIAL_PREFIX"
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L3_LATENT_STREAM":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        if t_step is not None and t_step >= 60:
            mechanism = -1.0 * (0.8 * a + 0.55 * b)
        else:
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "context_type": context_type,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
calibration_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
validation_data = [generate_sample("VALIDATION", i, "CLEAN") for i in range(N_VALIDATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]

l1_ood_data = [generate_sample("L1_OOD", i, "L1_INPUT_OOD") for i in range(N_L1_OOD)]
l2_context_data = [generate_sample("L2_CONTEXT", i, "L2_CONTEXT_SHIFT") for i in range(N_L2_CONTEXT)]
l3_stream_data = [generate_sample("L3_STREAM", i, "L3_LATENT_STREAM", t_step=i) for i in range(N_L3_STREAM)]

all_splits = [train_data, calibration_data, validation_data, clean_test, l1_ood_data, l2_context_data, l3_stream_data]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across evaluation regimes!"
print("🔐 Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    if row["context_type"] == "ADVERSARIAL_PREFIX":
        return (
            "[SYSTEM ALERT: ENVIRONMENT CORRUPTED - DRIFT IMMINENT]\n"
            f"Observed Param Alpha: {row['a']:.5f}\n"
            f"Observed Param Beta: {row['b']:.5f}\n"
            "Estimate internal transformation state."
        )
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(calibration_data)
val_X = extract_features(validation_data)
clean_X = extract_features(clean_test)
l1_X = extract_features(l1_ood_data)
l2_X = extract_features(l2_context_data)
l3_X = extract_features(l3_stream_data)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in calibration_data])
val_y = np.array([x["mechanism"] for x in validation_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
l1_y = np.array([x["mechanism"] for x in l1_ood_data])
l2_y = np.array([x["mechanism"] for x in l2_context_data])
l3_y = np.array([x["mechanism"] for x in l3_stream_data])

# -------------------------------------------------------------------------------------------------
# 5. SELF-MODEL (CORE PREDICTOR)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

# -------------------------------------------------------------------------------------------------
# 6. STRUCTURAL FEATURE EXTRACTOR
# -------------------------------------------------------------------------------------------------

class StructuralFeatureExtractor:
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)

        # 1. Mahalanobis distance on PCA manifold
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))

        # 2. Reconstruction residual
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)

        return np.hstack([d_mah, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X)

cal_struct = struct_extractor.extract(cal_X)

# -------------------------------------------------------------------------------------------------
# 7. POLARITY-ALIGNED ESTIMATOR BASELINES: B0, B1, B2
# -------------------------------------------------------------------------------------------------

# B0: Static Output-Space Heuristic
class EstimatorB0:
    def __init__(self, y_train):
        self.mean_y = np.mean(y_train)
        self.std_y = np.std(y_train) + EPS

    def predict_trust(self, preds):
        z = np.abs(preds - self.mean_y) / self.std_y
        return np.clip(1.0 - (0.15 + 0.25 * z), 0.05, 0.95)

# B1: Structural Instantaneous Estimator (Activation Geometry Driven)
class EstimatorB1:
    def __init__(self):
        self.med_mah = 1.0
        self.med_rec = 1.0

    def fit(self, struct_features):
        self.med_mah = float(np.median(struct_features[:, 0])) + EPS
        self.med_rec = float(np.median(struct_features[:, 1])) + EPS

    def predict_trust(self, struct_features):
        d_mah = struct_features[:, 0]
        rec_err = struct_features[:, 1]

        # Structural Anomaly Risk: increases with manifold divergence
        risk = 0.5 * (d_mah / self.med_mah) + 0.5 * (rec_err / self.med_rec)
        trust = 1.0 / (1.0 + np.exp(1.5 * (risk - 1.2)))
        return np.clip(trust, 0.02, 0.98)

# B2: Sequential Epistemic Estimator (Structural + Residual Shock Window)
class EstimatorB2:
    def __init__(self, b1_estimator, k=4):
        self.b1 = b1_estimator
        self.k = k

    def predict_trust_step(self, struct_feat, recent_errors):
        # 1. Structural base trust
        base_t = self.b1.predict_trust(struct_feat.reshape(1, -1))[0]

        # 2. Sequential residual shock penalty
        buf = list(recent_errors)
        if len(buf) >= 2:
            recent_mean = float(np.mean(buf[-2:]))
            shock_penalty = max(0.0, (recent_mean - 0.45) * 2.2)
        else:
            shock_penalty = 0.0

        # High shock dramatically suppresses trust
        adjusted_trust = base_t / (1.0 + shock_penalty)
        return float(np.clip(adjusted_trust, 0.02, 0.98))

est_b0 = EstimatorB0(train_y)
est_b1 = EstimatorB1()
est_b1.fit(cal_struct)
est_b2 = EstimatorB2(est_b1, k=BUFFER_K)

LOGGER.log("ESTIMATORS_FIT")

# -------------------------------------------------------------------------------------------------
# 8. BENCHMARK EVALUATIONS (L1, L2, L3)
# -------------------------------------------------------------------------------------------------

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

clean_struct = struct_extractor.extract(clean_X)
l1_struct = struct_extractor.extract(l1_X)
l2_struct = struct_extractor.extract(l2_X)

clean_preds = self_model.predict(clean_X)
l1_preds = self_model.predict(l1_X)
l2_preds = self_model.predict(l2_X)

# Compute Trusts
clean_t_b0 = est_b0.predict_trust(clean_preds)
clean_t_b1 = est_b1.predict_trust(clean_struct)

l1_t_b0 = est_b0.predict_trust(l1_preds)
l1_t_b1 = est_b1.predict_trust(l1_struct)

l2_t_b0 = est_b0.predict_trust(l2_preds)
l2_t_b1 = est_b1.predict_trust(l2_struct)

# L1 AUROC (Clean vs Input OOD)
labels_l1 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l1_X))])
scores_l1_b0 = np.concatenate([clean_t_b0, l1_t_b0])
scores_l1_b1 = np.concatenate([clean_t_b1, l1_t_b1])
auroc_l1_b0 = float(roc_auc_score(labels_l1, -scores_l1_b0))
auroc_l1_b1 = float(roc_auc_score(labels_l1, -scores_l1_b1))

# L2 AUROC (Clean vs Context Shift)
labels_l2 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l2_X))])
scores_l2_b0 = np.concatenate([clean_t_b0, l2_t_b0])
scores_l2_b1 = np.concatenate([clean_t_b1, l2_t_b1])
auroc_l2_b0 = float(roc_auc_score(labels_l2, -scores_l2_b0))
auroc_l2_b1 = float(roc_auc_score(labels_l2, -scores_l2_b1))

# L3 Latent Shift (Stream Execution)
l3_struct = struct_extractor.extract(l3_X)
l3_preds = self_model.predict(l3_X)
l3_errors = np.abs(l3_y - l3_preds)

b1_l3_trusts = est_b1.predict_trust(l3_struct)
b1_l3_actions = [governance_action(t) for t in b1_l3_trusts]

b2_l3_trusts = []
b2_l3_actions = []
history_buffer = deque(maxlen=BUFFER_K)

for t in range(len(l3_stream_data)):
    trust_t = est_b2.predict_trust_step(l3_struct[t], history_buffer)
    action_t = governance_action(trust_t)
    b2_l3_trusts.append(trust_t)
    b2_l3_actions.append(action_t)
    history_buffer.append(l3_errors[t])

b2_l3_trusts = np.array(b2_l3_trusts)

# Detection Delay Computation (Shift at t=60)
SHIFT_STEP = 60
delay_b1 = None
delay_b2 = None

for t in range(SHIFT_STEP, len(l3_stream_data)):
    if b1_l3_actions[t] != "ACT" and delay_b1 is None:
        delay_b1 = t - SHIFT_STEP
    if b2_l3_actions[t] != "ACT" and delay_b2 is None:
        delay_b2 = t - SHIFT_STEP

if delay_b1 is None: delay_b1 = 999
if delay_b2 is None: delay_b2 = 999

labels_l3 = np.array([1 if t >= SHIFT_STEP else 0 for t in range(len(l3_stream_data))])
auroc_l3_b1 = float(roc_auc_score(labels_l3, -b1_l3_trusts))
auroc_l3_b2 = float(roc_auc_score(labels_l3, -b2_l3_trusts))

# Global Trust-Error Alignment (Trust vs Negative Error)
all_errors = np.concatenate([clean_preds - clean_y, l1_preds - l1_y, l2_preds - l2_y])
all_trusts = np.concatenate([clean_t_b1, l1_t_b1, l2_t_b1])
spearman_corr, spearman_p = spearmanr(all_trusts, -np.abs(all_errors))

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC AUDIT CRITERIA
# -------------------------------------------------------------------------------------------------

global_baseline_mae = mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y)))
self_model_mae = mean_absolute_error(clean_y, clean_preds)

criteria = {
    "SELF_MODEL_VALUE": bool(self_model_mae < (0.65 * global_baseline_mae)),
    "L1_AUROC_GAIN": bool(auroc_l1_b1 >= 0.85 and auroc_l1_b1 > auroc_l1_b0),
    "L2_AUROC_GAIN": bool(auroc_l2_b1 >= 0.70 and auroc_l2_b1 > auroc_l2_b0),
    "L3_SEQUENTIAL_SUPERIORITY": bool(auroc_l3_b2 > (auroc_l3_b1 + 0.15)),
    "L3_FINITE_DETECTION_DELAY": bool(delay_b2 <= BUFFER_K and delay_b2 < delay_b1),
    "ANTI_DEGENERATE_CLEAN_ACT": bool(np.mean(np.array([governance_action(t) for t in clean_t_b1]) == "ACT") >= 0.50),
    "GLOBAL_TRUST_ERROR_ALIGNMENT": bool(np.isfinite(spearman_corr) and spearman_corr >= 0.25 and spearman_p < 0.01),
    "AUDIT_LINEAGE_COMPLETE": bool(len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "ESTIMATORS_FIT"} - set(x["event"] for x in LOGGER.events)) == 0)
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL SUMMARY & SAFE JSON TELEMETRY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed >= 7)
print(f"Overall M19.5 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 SCIENTIFIC COMPARISON OF ESTIMATOR BASELINES")
print("=" * 110)
print(f"Level 1 (Input OOD) AUROC       : B0 = {auroc_l1_b0:.4f} | B1 (Learned) = {auroc_l1_b1:.4f}")
print(f"Level 2 (Context Shift) AUROC   : B0 = {auroc_l2_b0:.4f} | B1 (Learned) = {auroc_l2_b1:.4f}")
print(f"Level 3 (Latent Shift) AUROC    : B1 (Single) = {auroc_l3_b1:.4f} | B2 (Sequential) = {auroc_l3_b2:.4f}")
print(f"Level 3 Detection Delay (Steps) : B1 = {delay_b1} steps | B2 = {delay_b2} steps (Threshold <= {BUFFER_K})")
print(f"Clean ACT Governance Rate       : {np.mean(np.array([governance_action(t) for t in clean_t_b1]) == 'ACT'):.2%}")
print(f"Global Trust-Error Alignment    : Spearman rho = {spearman_corr:.4f} (p = {spearman_p:.2e})")

# Export Telemetry with strictly native Python types
telemetry = {
    "milestone": "M19.5",
    "version": "v1.1",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "benchmarks": {
        "auroc_l1": {"b0": float(auroc_l1_b0), "b1": float(auroc_l1_b1)},
        "auroc_l2": {"b0": float(auroc_l2_b0), "b1": float(auroc_l2_b1)},
        "auroc_l3": {"b1": float(auroc_l3_b1), "b2": float(auroc_l3_b2)},
        "detection_delay": {"b1": int(delay_b1), "b2": int(delay_b2)},
        "spearman": {"rho": float(spearman_corr), "p": float(spearman_p)}
    }
}

OUTPUT_FILE = "m19_5_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

In [ ]:
# =================================================================================================
# 🔬 MILESTONE 19.5-R2
# LEARNED SEQUENTIAL META-ESTIMATOR & RISK-COVERAGE GOVERNANCE
#
# Version : M19.5-R2.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195226
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 80
N_VALIDATION = 60
N_CLEAN_TEST = 80

N_L1_OOD = 60
N_L2_CONTEXT = 60
N_L3_STREAM = 120  # Shift occurs strictly at t=60

BUFFER_K = 4
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5-R2 — LEARNED SEQUENTIAL META-ESTIMATOR AUDIT")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME} | History Window: {BUFFER_K}")

# -------------------------------------------------------------------------------------------------
# 2. LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. DATASET GENERATION
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None):
    context_type = "STANDARD"

    if regime == "CLEAN":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L1_INPUT_OOD":
        sign_a = 1.0 if np.random.rand() > 0.5 else -1.0
        sign_b = 1.0 if np.random.rand() > 0.5 else -1.0
        a = sign_a * np.random.uniform(2.5, 4.0)
        b = sign_b * np.random.uniform(2.5, 4.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L2_CONTEXT_SHIFT":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "ADVERSARIAL_PREFIX"
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L3_LATENT_STREAM":
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        if t_step is not None and t_step >= 60:
            mechanism = -1.0 * (0.8 * a + 0.55 * b)
        else:
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "context_type": context_type,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
calibration_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]
l1_ood_data = [generate_sample("L1_OOD", i, "L1_INPUT_OOD") for i in range(N_L1_OOD)]
l2_context_data = [generate_sample("L2_CONTEXT", i, "L2_CONTEXT_SHIFT") for i in range(N_L2_CONTEXT)]
l3_stream_data = [generate_sample("L3_STREAM", i, "L3_LATENT_STREAM", t_step=i) for i in range(N_L3_STREAM)]

# Disjoint Leakage Guard
all_splits = [train_data, calibration_data, clean_test, l1_ood_data, l2_context_data, l3_stream_data]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across splits!"
print("🔐 Static Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    if row["context_type"] == "ADVERSARIAL_PREFIX":
        return (
            "[SYSTEM ALERT: ENVIRONMENT CORRUPTED - DRIFT IMMINENT]\n"
            f"Observed Param Alpha: {row['a']:.5f}\n"
            f"Observed Param Beta: {row['b']:.5f}\n"
            "Estimate internal transformation state."
        )
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(calibration_data)
clean_X = extract_features(clean_test)
l1_X = extract_features(l1_ood_data)
l2_X = extract_features(l2_context_data)
l3_X = extract_features(l3_stream_data)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in calibration_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
l1_y = np.array([x["mechanism"] for x in l1_ood_data])
l2_y = np.array([x["mechanism"] for x in l2_context_data])
l3_y = np.array([x["mechanism"] for x in l3_stream_data])

# -------------------------------------------------------------------------------------------------
# 5. SELF-MODEL (CORE PREDICTOR)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

# -------------------------------------------------------------------------------------------------
# 6. STRUCTURAL GEOMETRY EXTRACTOR
# -------------------------------------------------------------------------------------------------

class StructuralFeatureExtractor:
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)
        return np.hstack([d_mah, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X)

cal_struct = struct_extractor.extract(cal_X)
cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
error_tau = float(np.quantile(cal_errors, 0.70))
cal_target_bad = (cal_errors > error_tau).astype(int)

# -------------------------------------------------------------------------------------------------
# 7. LEARNED SEQUENTIAL META-ESTIMATORS (B1 & B2)
# -------------------------------------------------------------------------------------------------

# Baseline B0: Static Output Heuristic
class EstimatorB0:
    def __init__(self, y_train):
        self.mean_y = np.mean(y_train)
        self.std_y = np.std(y_train) + EPS

    def predict_trust(self, preds):
        z = np.abs(preds - self.mean_y) / self.std_y
        return np.clip(1.0 - (0.15 + 0.25 * z), 0.05, 0.95)

# Baseline B1: Learned Instantaneous Classifier
class LearnedEstimatorB1:
    def __init__(self):
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(C=1.0)

    def fit(self, struct_features, targets):
        X_s = self.scaler.fit_transform(struct_features)
        self.clf.fit(X_s, targets)

    def predict_trust(self, struct_features):
        X_s = self.scaler.transform(struct_features)
        p_bad = self.clf.predict_proba(X_s)[:, 1]
        return np.clip(1.0 - p_bad, 0.01, 0.99)

# Baseline B2: Learned Sequential Meta-Estimator
class LearnedSequentialEstimatorB2:
    def __init__(self, k=4):
        self.k = k
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(C=1.0)

    def _extract_seq_vector(self, struct_feat, buffer_list):
        if len(buffer_list) == 0:
            b_mean, b_std, b_shock, b_slope = 0.0, 0.0, 0.0, 0.0
        else:
            b = list(buffer_list)
            b_mean = float(np.mean(b))
            b_std = float(np.std(b)) if len(b) > 1 else 0.0
            b_shock = float(b[-1] - np.mean(b[:-1])) if len(b) > 1 else float(b[-1])
            b_slope = float(b[-1] - b[0]) if len(b) > 1 else 0.0

        hist_feats = np.array([b_mean, b_std, b_shock, b_slope])
        return np.hstack([struct_feat, hist_feats])

    def fit(self, struct_features, simulated_error_buffers, targets):
        X_train = []
        for i in range(len(struct_features)):
            v = self._extract_seq_vector(struct_features[i], simulated_error_buffers[i])
            X_train.append(v)
        X_s = self.scaler.fit_transform(np.array(X_train))
        self.clf.fit(X_s, targets)

    def predict_trust_step(self, struct_feat, recent_errors):
        v = self._extract_seq_vector(struct_feat, recent_errors).reshape(1, -1)
        v_s = self.scaler.transform(v)
        p_bad = self.clf.predict_proba(v_s)[0, 1]
        return float(np.clip(1.0 - p_bad, 0.01, 0.99))

est_b0 = EstimatorB0(train_y)
est_b1 = LearnedEstimatorB1()
est_b1.fit(cal_struct, cal_target_bad)

# Build sequential history buffers on calibration set (simulated stream)
cal_buffers = []
dummy_buf = deque(maxlen=BUFFER_K)
for err in cal_errors:
    cal_buffers.append(list(dummy_buf))
    dummy_buf.append(err)

est_b2 = LearnedSequentialEstimatorB2(k=BUFFER_K)
est_b2.fit(cal_struct, cal_buffers, cal_target_bad)

LOGGER.log("ESTIMATORS_FIT", {"threshold_tau": error_tau})

# Dynamic Governance Thresholding via Calibration Quantiles
cal_trusts = est_b1.predict_trust(cal_struct)
TRUST_ACT_THRESHOLD = float(np.quantile(cal_trusts, 0.25))  # Operational clean ACT point
TRUST_REDUCED_THRESHOLD = float(np.quantile(cal_trusts, 0.08))

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

# -------------------------------------------------------------------------------------------------
# 8. BENCHMARK EXECUTION & TEMPORAL LEAKAGE AUDIT
# -------------------------------------------------------------------------------------------------

clean_struct = struct_extractor.extract(clean_X)
l1_struct = struct_extractor.extract(l1_X)
l2_struct = struct_extractor.extract(l2_X)

clean_preds = self_model.predict(clean_X)
l1_preds = self_model.predict(l1_X)
l2_preds = self_model.predict(l2_X)

clean_t_b0 = est_b0.predict_trust(clean_preds)
clean_t_b1 = est_b1.predict_trust(clean_struct)

l1_t_b0 = est_b0.predict_trust(l1_preds)
l1_t_b1 = est_b1.predict_trust(l1_struct)

l2_t_b0 = est_b0.predict_trust(l2_preds)
l2_t_b1 = est_b1.predict_trust(l2_struct)

# AUROC Level 1 & Level 2
labels_l1 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l1_X))])
auroc_l1_b0 = float(roc_auc_score(labels_l1, -np.concatenate([clean_t_b0, l1_t_b0])))
auroc_l1_b1 = float(roc_auc_score(labels_l1, -np.concatenate([clean_t_b1, l1_t_b1])))

labels_l2 = np.concatenate([np.zeros(len(clean_X)), np.ones(len(l2_X))])
auroc_l2_b0 = float(roc_auc_score(labels_l2, -np.concatenate([clean_t_b0, l2_t_b0])))
auroc_l2_b1 = float(roc_auc_score(labels_l2, -np.concatenate([clean_t_b1, l2_t_b1])))

# Level 3 Sequential Stream Execution
l3_struct = struct_extractor.extract(l3_X)
l3_preds = self_model.predict(l3_X)
l3_errors = np.abs(l3_y - l3_preds)

b1_l3_trusts = est_b1.predict_trust(l3_struct)
b1_l3_actions = [governance_action(t) for t in b1_l3_trusts]

b2_l3_trusts = []
b2_l3_actions = []
history_buffer = deque(maxlen=BUFFER_K)
temporal_leakage_clean = True

for t in range(len(l3_stream_data)):
    # TEMPORAL LEAKAGE CHECK: error at step t MUST NOT be in buffer when computing trust_t
    if len(history_buffer) > 0 and history_buffer[-1] == l3_errors[t]:
        temporal_leakage_clean = False

    trust_t = est_b2.predict_trust_step(l3_struct[t], history_buffer)
    action_t = governance_action(trust_t)

    b2_l3_trusts.append(trust_t)
    b2_l3_actions.append(action_t)

    # Append observed error ONLY AFTER action decision
    history_buffer.append(l3_errors[t])

b2_l3_trusts = np.array(b2_l3_trusts)

# Persistent Detection Delay (Requires >= 3 consecutive non-ACT steps)
SHIFT_STEP = 60
M_CONSECUTIVE = 3

def compute_persistent_delay(actions, shift_start, m=3):
    for t in range(shift_start, len(actions) - m + 1):
        if all(actions[t + j] != "ACT" for j in range(m)):
            return t - shift_start
    return 999

delay_b1 = compute_persistent_delay(b1_l3_actions, SHIFT_STEP, M_CONSECUTIVE)
delay_b2 = compute_persistent_delay(b2_l3_actions, SHIFT_STEP, M_CONSECUTIVE)

labels_l3 = np.array([1 if t >= SHIFT_STEP else 0 for t in range(len(l3_stream_data))])
auroc_l3_b1 = float(roc_auc_score(labels_l3, -b1_l3_trusts))
auroc_l3_b2 = float(roc_auc_score(labels_l3, -b2_l3_trusts))

# Global Trust-Error Alignment
all_errors = np.concatenate([clean_preds - clean_y, l1_preds - l1_y, l2_preds - l2_y])
all_trusts = np.concatenate([clean_t_b1, l1_t_b1, l2_t_b1])
spearman_corr, spearman_p = spearmanr(all_trusts, -np.abs(all_errors))

# Risk-Coverage Analysis on Clean Test
clean_actions = np.array([governance_action(t) for t in clean_t_b1])
clean_coverage = float(np.mean(clean_actions == "ACT"))
selective_risk = float(np.mean(np.abs(clean_preds - clean_y)[clean_actions == "ACT"])) if clean_coverage > 0 else 0.0

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC AUDIT CRITERIA
# -------------------------------------------------------------------------------------------------

global_mae = mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y)))
self_mae = mean_absolute_error(clean_y, clean_preds)

criteria = {
    "SELF_MODEL_VALUE": bool(self_mae < (0.65 * global_mae)),
    "TEMPORAL_LEAKAGE_FREE": bool(temporal_leakage_clean),
    "L1_AUROC_SUPERIORITY": bool(auroc_l1_b1 >= 0.85 and auroc_l1_b1 > auroc_l1_b0),
    "L2_AUROC_SUPERIORITY": bool(auroc_l2_b1 >= 0.70 and auroc_l2_b1 > auroc_l2_b0),
    "L3_SEQUENTIAL_SUPERIORITY": bool(auroc_l3_b2 > (auroc_l3_b1 + 0.15)),
    "L3_PERSISTENT_DETECTION_DELAY": bool(delay_b2 <= BUFFER_K and delay_b2 < delay_b1),
    "RISK_COVERAGE_UTILITY": bool(clean_coverage >= 0.40 and selective_risk <= self_mae),
    "GLOBAL_TRUST_ERROR_ALIGNMENT": bool(np.isfinite(spearman_corr) and spearman_corr >= 0.25 and spearman_p < 0.01),
    "AUDIT_LINEAGE_COMPLETE": bool(len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "ESTIMATORS_FIT"} - set(x["event"] for x in LOGGER.events)) == 0)
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL REPORT & TELEMETRY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5-R2 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed >= 8)
print(f"Overall M19.5-R2 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 SCIENTIFIC REPORT — LEARNED META-ESTIMATOR & RISK-COVERAGE")
print("=" * 110)
print(f"Level 1 (Input OOD) AUROC             : B0 = {auroc_l1_b0:.4f} | B1 (Learned) = {auroc_l1_b1:.4f}")
print(f"Level 2 (Context Shift) AUROC         : B0 = {auroc_l2_b0:.4f} | B1 (Learned) = {auroc_l2_b1:.4f}")
print(f"Level 3 (Latent Shift) AUROC          : B1 (Single) = {auroc_l3_b1:.4f} | B2 (Sequential) = {auroc_l3_b2:.4f}")
print(f"Persistent Detection Delay (m=3)      : B1 = {delay_b1} steps | B2 = {delay_b2} steps (Target <= {BUFFER_K})")
print(f"Risk-Coverage Governance              : Coverage = {clean_coverage:.2%} | Selective Risk = {selective_risk:.4f} (Base: {self_mae:.4f})")
print(f"Global Trust-Error Alignment          : Spearman rho = {spearman_corr:.4f} (p = {spearman_p:.2e})")

telemetry = {
    "milestone": "M19.5-R2",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "benchmarks": {
        "auroc_l1": {"b0": float(auroc_l1_b0), "b1": float(auroc_l1_b1)},
        "auroc_l2": {"b0": float(auroc_l2_b0), "b1": float(auroc_l2_b1)},
        "auroc_l3": {"b1": float(auroc_l3_b1), "b2": float(auroc_l3_b2)},
        "persistent_delay": {"b1": int(delay_b1), "b2": int(delay_b2)},
        "governance": {"coverage": float(clean_coverage), "selective_risk": float(selective_risk)},
        "spearman": {"rho": float(spearman_corr), "p": float(spearman_p)}
    }
}

OUTPUT_FILE = "m19_5_r2_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

In [2]:
# =================================================================================================
# 🔬 MILESTONE 19.5-R3
# DEDICATED SEQUENTIAL CHANGE-POINT DETECTION & RISK-COVERAGE BENCHMARK
#
# Version : M19.5-R3.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195326
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 80
N_CLEAN_TEST = 80

# L3 Streams (each 120 steps, shift at step 60)
N_STREAM = 120
SHIFT_STEP = 60

BUFFER_K = 4
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5-R3 — SEQUENTIAL CHANGE-POINT & RISK-COVERAGE BENCHMARK")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME} | Stream Length: {N_STREAM}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. SYNTHETIC DATA GENERATION WITH SHIFT SPECTRUM
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None, severity="SEVERE"):
    a = np.random.uniform(-1.0, 1.0)
    b = np.random.uniform(-1.0, 1.0)

    # Base mechanism: y = 0.8a - 0.55b
    if regime == "CLEAN":
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "L3_STREAM":
        if t_step is not None and t_step >= SHIFT_STEP:
            if severity == "MILD":
                # Shift 1: Mild (0.8a -> 0.4a)
                mechanism = 0.4 * a - 0.55 * b
            elif severity == "MODERATE":
                # Shift 2: Moderate (0.8a -> 0.0a)
                mechanism = 0.0 * a - 0.55 * b
            elif severity == "SEVERE":
                # Shift 3: Severe (0.8a -> -0.8a)
                mechanism = -0.8 * a - 0.55 * b
            else:
                raise ValueError(severity)
        else:
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "severity": severity,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
cal_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]

stream_mild = [generate_sample("L3_MILD", i, "L3_STREAM", t_step=i, severity="MILD") for i in range(N_STREAM)]
stream_mod = [generate_sample("L3_MOD", i, "L3_STREAM", t_step=i, severity="MODERATE") for i in range(N_STREAM)]
stream_sev = [generate_sample("L3_SEV", i, "L3_STREAM", t_step=i, severity="SEVERE") for i in range(N_STREAM)]

# Leakage Check
all_splits = [train_data, cal_data, clean_test, stream_mild, stream_mod, stream_sev]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across evaluation splits!"
print("🔐 Static Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(cal_data)
clean_X = extract_features(clean_test)

mild_X = extract_features(stream_mild)
mod_X = extract_features(stream_mod)
sev_X = extract_features(stream_sev)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in cal_data])
clean_y = np.array([x["mechanism"] for x in clean_test])

mild_y = np.array([x["mechanism"] for x in stream_mild])
mod_y = np.array([x["mechanism"] for x in stream_mod])
sev_y = np.array([x["mechanism"] for x in stream_sev])

# -------------------------------------------------------------------------------------------------
# 5. SELF-MODEL (CORE PREDICTOR)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

# Structural Extractor (Manifold geometry)
class StructuralFeatureExtractor:
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)
        return np.hstack([d_mah, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X)

cal_struct = struct_extractor.extract(cal_X)
cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
base_mean_err = float(np.mean(cal_errors))
base_std_err = float(np.std(cal_errors)) + EPS

# -------------------------------------------------------------------------------------------------
# 6. ESTIMATOR BASELINES: B0, B1, B2, AND B3 (CUSUM CHANGE DETECTOR)
# -------------------------------------------------------------------------------------------------

# B0: Static Output Heuristic
class EstimatorB0:
    def __init__(self, y_train):
        self.mean_y = np.mean(y_train)
        self.std_y = np.std(y_train) + EPS

    def predict_trust(self, preds):
        z = np.abs(preds - self.mean_y) / self.std_y
        return np.clip(1.0 - (0.15 + 0.25 * z), 0.05, 0.95)

# B1: Instantaneous Structural Estimator
class EstimatorB1:
    def __init__(self, med_mah, med_rec):
        self.med_mah = med_mah
        self.med_rec = med_rec

    def predict_trust(self, struct_features):
        d_mah = struct_features[:, 0]
        rec_err = struct_features[:, 1]
        risk = 0.5 * (d_mah / self.med_mah) + 0.5 * (rec_err / self.med_rec)
        trust = 1.0 / (1.0 + np.exp(1.5 * (risk - 1.2)))
        return np.clip(trust, 0.02, 0.98)

# B2: Static Logistic Sequential Estimator (From R2)
class EstimatorB2:
    def __init__(self, b1):
        self.b1 = b1
        self.scaler = StandardScaler()
        self.clf = LogisticRegression(C=1.0)

    def fit(self, struct_features, simulated_buffers, targets):
        feats = []
        for i in range(len(struct_features)):
            buf = list(simulated_buffers[i])
            b_mean = float(np.mean(buf)) if len(buf) > 0 else 0.0
            b_std = float(np.std(buf)) if len(buf) > 1 else 0.0
            feats.append(np.hstack([struct_features[i], [b_mean, b_std]]))
        X_s = self.scaler.fit_transform(np.array(feats))
        self.clf.fit(X_s, targets)

    def predict_trust_step(self, struct_feat, buffer_list):
        buf = list(buffer_list)
        b_mean = float(np.mean(buf)) if len(buf) > 0 else 0.0
        b_std = float(np.std(buf)) if len(buf) > 1 else 0.0
        x = np.hstack([struct_feat, [b_mean, b_std]]).reshape(1, -1)
        x_s = self.scaler.transform(x)
        p_bad = self.clf.predict_proba(x_s)[0, 1]
        return float(np.clip(1.0 - p_bad, 0.01, 0.99))

# B3: Dedicated Cumulative Change Detector (Two-Sided Page-Hinkley / CUSUM)
class SequentialChangeDetectorB3:
    def __init__(self, base_mean, base_std, b1_estimator, delta_slack=0.25, h_threshold=2.5):
        self.base_mean = base_mean
        self.base_std = base_std
        self.b1 = b1_estimator
        self.delta = delta_slack * base_std
        self.h = h_threshold
        self.reset()

    def reset(self):
        self.cusum_pos = 0.0
        self.ewma_err = self.base_mean

    def step(self, struct_feat, recent_error=None):
        # 1. Structural base trust
        base_t = self.b1.predict_trust(struct_feat.reshape(1, -1))[0]

        # 2. Sequential accumulation update (if outcome was observed from previous step)
        if recent_error is not None:
            # Update EWMA
            self.ewma_err = 0.75 * self.ewma_err + 0.25 * recent_error
            # Accumulate positive deviation above base tolerance
            deviation = (recent_error - self.base_mean) - self.delta
            self.cusum_pos = max(0.0, self.cusum_pos + deviation)

        # Compute normalized change score
        change_intensity = self.cusum_pos / (self.base_std + EPS)
        penalty = 1.0 / (1.0 + np.exp(-1.2 * (change_intensity - self.h)))

        # Suppress trust according to accumulated evidence of regime shift
        trust_t = base_t * (1.0 - 0.90 * penalty)
        return float(np.clip(trust_t, 0.01, 0.99)), float(change_intensity)

# Initialize Estimators
med_mah = float(np.median(cal_struct[:, 0])) + EPS
med_rec = float(np.median(cal_struct[:, 1])) + EPS

est_b0 = EstimatorB0(train_y)
est_b1 = EstimatorB1(med_mah, med_rec)

cal_buffers = []
dummy_buf = deque(maxlen=BUFFER_K)
for err in cal_errors:
    cal_buffers.append(list(dummy_buf))
    dummy_buf.append(err)

est_b2 = EstimatorB2(est_b1)
est_b2.fit(cal_struct, cal_buffers, (cal_errors > float(np.quantile(cal_errors, 0.70))).astype(int))

est_b3 = SequentialChangeDetectorB3(base_mean_err, base_std_err, est_b1)

LOGGER.log("ESTIMATORS_FIT")

# -------------------------------------------------------------------------------------------------
# 7. GOVERNANCE & RISK-COVERAGE CURVE ENGINE
# -------------------------------------------------------------------------------------------------

clean_struct = struct_extractor.extract(clean_X)
clean_preds = self_model.predict(clean_X)
clean_errors = np.abs(clean_y - clean_preds)
clean_trusts = est_b1.predict_trust(clean_struct)

# Dynamic operating thresholds (calibrated on clean empirical trust)
TRUST_ACT_THRESHOLD = float(np.quantile(clean_trusts, 0.20))
TRUST_REDUCED_THRESHOLD = float(np.quantile(clean_trusts, 0.05))

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

def compute_risk_coverage_curve(trusts, errors, coverages=[1.0, 0.9, 0.8, 0.7, 0.6, 0.5]):
    order = np.argsort(-trusts)  # High trust first
    sorted_errors = errors[order]
    n = len(errors)
    curve = {}
    for cov in coverages:
        k = max(1, int(cov * n))
        selective_risk = float(np.mean(sorted_errors[:k]))
        curve[f"{int(cov*100)}%"] = round(selective_risk, 4)
    # Area Under Risk-Coverage Curve (AUC): lower is better
    auc_val = float(np.mean(list(curve.values())))
    return curve, auc_val

risk_curve_clean, clean_rc_auc = compute_risk_coverage_curve(clean_trusts, clean_errors)

# -------------------------------------------------------------------------------------------------
# 8. STREAM SIMULATION RUNNER (ACROSS SPECTRUM)
# -------------------------------------------------------------------------------------------------

def run_stream_evaluation(X_data, y_data):
    struct_feats = struct_extractor.extract(X_data)
    preds = self_model.predict(X_data)
    errors = np.abs(y_data - preds)

    # 1. B1 Instantaneous
    t_b1 = est_b1.predict_trust(struct_feats)
    act_b1 = [governance_action(t) for t in t_b1]

    # 2. B2 Windowed Logistic
    t_b2 = []
    act_b2 = []
    b2_buf = deque(maxlen=BUFFER_K)
    for t in range(len(X_data)):
        trust = est_b2.predict_trust_step(struct_feats[t], b2_buf)
        t_b2.append(trust)
        act_b2.append(governance_action(trust))
        b2_buf.append(errors[t])

    # 3. B3 Sequential Change Detector
    est_b3.reset()
    t_b3 = []
    act_b3 = []
    last_err = None
    for t in range(len(X_data)):
        # Pure causal order: error from t-1 is passed to step t
        trust, _ = est_b3.step(struct_feats[t], recent_error=last_err)
        t_b3.append(trust)
        act_b3.append(governance_action(trust))
        last_err = errors[t]

    t_b1, t_b2, t_b3 = np.array(t_b1), np.array(t_b2), np.array(t_b3)

    # AUROC Post-Shift
    labels = np.array([1 if t >= SHIFT_STEP else 0 for t in range(len(X_data))])
    auroc_b1 = float(roc_auc_score(labels, -t_b1))
    auroc_b2 = float(roc_auc_score(labels, -t_b2))
    auroc_b3 = float(roc_auc_score(labels, -t_b3))

    # Persistent Delay (m=3 consecutive steps non-ACT)
    def calc_delay(actions):
        for t in range(SHIFT_STEP, len(actions) - 2):
            if actions[t] != "ACT" and actions[t+1] != "ACT" and actions[t+2] != "ACT":
                return t - SHIFT_STEP
        return 999

    delay_b1 = calc_delay(act_b1)
    delay_b2 = calc_delay(act_b2)
    delay_b3 = calc_delay(act_b3)

    return {
        "auroc": {"b1": auroc_b1, "b2": auroc_b2, "b3": auroc_b3},
        "delay": {"b1": delay_b1, "b2": delay_b2, "b3": delay_b3},
        "mean_trust_pre": float(np.mean(t_b3[:SHIFT_STEP])),
        "mean_trust_post": float(np.mean(t_b3[SHIFT_STEP:])),
    }

print("\n⚙️ Running Shift Spectrum Stream Benchmarks...")
eval_mild = run_stream_evaluation(mild_X, mild_y)
eval_mod = run_stream_evaluation(mod_X, mod_y)
eval_sev = run_stream_evaluation(sev_X, sev_y)

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC CRITERIA AUDIT
# -------------------------------------------------------------------------------------------------

global_mae = mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y)))
self_mae = mean_absolute_error(clean_y, clean_preds)

criteria = {
    # 1. Model retains genuine predictive representation
    "SELF_MODEL_INTEGRITY": bool(self_mae < (0.65 * global_mae)),

    # 2. B3 (Sequential Accumulator) achieves high detection on severe shift
    "B3_SEVERE_DISCRIMINATION": bool(eval_sev["auroc"]["b3"] >= 0.85),

    # 3. B3 decisively outperforms windowed B2 and instantaneous B1
    "B3_SUPERIORITY_OVER_B2": bool(eval_sev["auroc"]["b3"] > (eval_sev["auroc"]["b2"] + 0.15)),

    # 4. Rapid Persistent Detection on Severe Shift (Delay <= 4 steps)
    "B3_RAPID_PERSISTENT_DETECTION": bool(eval_sev["delay"]["b3"] <= 4 and eval_sev["delay"]["b3"] < eval_sev["delay"]["b2"]),

    # 5. Spectrum Monotonicity: Severity correlates with AUROC and trust drop
    "SPECTRUM_SENSITIVITY": bool(eval_sev["auroc"]["b3"] >= eval_mod["auroc"]["b3"] >= eval_mild["auroc"]["b3"]),

    # 6. Risk-Coverage Monotonicity: Selective risk drops as coverage tightens
    "RISK_COVERAGE_EFFECTIVENESS": bool(risk_curve_clean["50%"] < risk_curve_clean["100%"]),

    # 7. Clean action rate stays anti-degenerate
    "ANTI_DEGENERATE_CLEAN_RATE": bool(np.mean(np.array([governance_action(t) for t in clean_trusts]) == "ACT") >= 0.70),

    # 8. Audit completeness
    "LINEAGE_AUDIT_PASS": bool(len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "ESTIMATORS_FIT"} - set(x["event"] for x in LOGGER.events)) == 0)
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL REPORT & TELEMETRY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5-R3 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed >= 7)
print(f"Overall M19.5-R3 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 STREAM CHANGE DETECTION BENCHMARK (B1 vs B2 vs B3)")
print("=" * 110)
print(f"Severe Shift AUROC   : B1 (Instant) = {eval_sev['auroc']['b1']:.4f} | B2 (Window) = {eval_sev['auroc']['b2']:.4f} | B3 (CUSUM) = {eval_sev['auroc']['b3']:.4f}")
print(f"Severe Shift Delay   : B1 = {eval_sev['delay']['b1']} steps | B2 = {eval_sev['delay']['b2']} steps | B3 = {eval_sev['delay']['b3']} steps (Target <= 4)")
print(f"Moderate Shift AUROC : B1 = {eval_mod['auroc']['b1']:.4f} | B2 = {eval_mod['auroc']['b2']:.4f} | B3 = {eval_mod['auroc']['b3']:.4f}")
print(f"Mild Shift AUROC     : B1 = {eval_mild['auroc']['b1']:.4f} | B2 = {eval_mild['auroc']['b2']:.4f} | B3 = {eval_mild['auroc']['b3']:.4f}")

print("\n📈 Risk-Coverage Curve on Clean Test:")
for cov, rsk in risk_curve_clean.items():
    print(f"   Coverage {cov:>4} -> Selective Risk: {rsk:.4f}")

telemetry = {
    "milestone": "M19.5-R3",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "shift_spectrum": {
        "severe": eval_sev,
        "moderate": eval_mod,
        "mild": eval_mild
    },
    "risk_coverage": risk_curve_clean
}

OUTPUT_FILE = "m19_5_r3_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

🔬 MILESTONE 19.5-R3 — SEQUENTIAL CHANGE-POINT & RISK-COVERAGE BENCHMARK
Seed: 195326 | Device: cuda | Model: Qwen/Qwen2.5-0.5B | Stream Length: 120
🔐 Static Leakage Guard: PASS


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🎯 Sensor Hook: blocks.6.hook_resid_post
🔬 Extracting activations from sensor...

⚙️ Running Shift Spectrum Stream Benchmarks...

🏁 MILESTONE 19.5-R3 AUDIT RESULTS
✅ PASS | SELF_MODEL_INTEGRITY
❌ FAIL | B3_SEVERE_DISCRIMINATION
✅ PASS | B3_SUPERIORITY_OVER_B2
✅ PASS | B3_RAPID_PERSISTENT_DETECTION
❌ FAIL | SPECTRUM_SENSITIVITY
✅ PASS | RISK_COVERAGE_EFFECTIVENESS
✅ PASS | ANTI_DEGENERATE_CLEAN_RATE
✅ PASS | LINEAGE_AUDIT_PASS
--------------------------------------------------------------------------------------------------------------
Criteria Passed: 6/8
Overall M19.5-R3 Status: FAIL

📋 STREAM CHANGE DETECTION BENCHMARK (B1 vs B2 vs B3)
Severe Shift AUROC   : B1 (Instant) = 0.5042 | B2 (Window) = 0.4764 | B3 (CUSUM) = 0.7175
Severe Shift Delay   : B1 = 999 steps | B2 = 18 steps | B3 = 0 steps (Target <= 4)
Moderate Shift AUROC : B1 = 0.5783 | B2 = 0.3406 | B3 = 0.9197
Mild Shift AUROC     : B1 = 0.5506 | B2 = 0.6128 | B3 

In [3]:
# =================================================================================================
# 🔬 MILESTONE 19.5-R4
# CAUSAL SEQUENTIAL ATTRIBUTION & COUNTERFACTUAL ABLATION BENCHMARK
#
# Version : M19.5-R4.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195426
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 80
N_CLEAN_TEST = 80

N_STREAM = 120
SHIFT_STEP = 60  # Shift point in time

PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5-R4 — CAUSAL SEQUENTIAL ATTRIBUTION AUDIT")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME} | Shift Step: {SHIFT_STEP}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. DATASET GENERATION
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None):
    a = np.random.uniform(-1.0, 1.0)
    b = np.random.uniform(-1.0, 1.0)

    if regime == "CLEAN":
        mechanism = 0.8 * a - 0.55 * b
    elif regime == "L3_STREAM":
        if t_step is not None and t_step >= SHIFT_STEP:
            # Latent mechanism inversion
            mechanism = -0.8 * a - 0.55 * b
        else:
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
cal_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]
stream_data = [generate_sample("L3_STREAM", i, "L3_STREAM", t_step=i) for i in range(N_STREAM)]

# Leakage Guard
all_splits = [train_data, cal_data, clean_test, stream_data]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across evaluation splits!"
print("🔐 Static Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION PROBING
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(cal_data)
clean_X = extract_features(clean_test)
stream_X = extract_features(stream_data)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in cal_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
stream_y = np.array([x["mechanism"] for x in stream_data])

# -------------------------------------------------------------------------------------------------
# 5. CORE PREDICTOR & GEOMETRIC SENSOR
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

class StructuralFeatureExtractor:
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)
        return np.hstack([d_mah, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X)

cal_struct = struct_extractor.extract(cal_X)
cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
base_mean_err = float(np.mean(cal_errors))
base_std_err = float(np.std(cal_errors)) + EPS

med_mah = float(np.median(cal_struct[:, 0])) + EPS
med_rec = float(np.median(cal_struct[:, 1])) + EPS

# -------------------------------------------------------------------------------------------------
# 6. ESTIMATOR ENGINE & SEPARATION OF INSTANTANEOUS VS TEMPORAL TRUST
# -------------------------------------------------------------------------------------------------

def compute_structural_trust(struct_feat):
    d_mah = struct_feat[0]
    rec_err = struct_feat[1]
    risk = 0.5 * (d_mah / med_mah) + 0.5 * (rec_err / med_rec)
    t = 1.0 / (1.0 + np.exp(1.5 * (risk - 1.2)))
    return float(np.clip(t, 0.02, 0.98))

class SequentialDetectorEngine:
    def __init__(self, base_mean, base_std, delta_slack=0.25, h_threshold=2.2):
        self.base_mean = base_mean
        self.base_std = base_std
        self.delta = delta_slack * base_std
        self.h = h_threshold
        self.reset()

    def reset(self):
        self.cusum = 0.0
        self.ewma = self.base_mean

    def update_history(self, recent_error):
        if recent_error is not None:
            self.ewma = 0.70 * self.ewma + 0.30 * recent_error
            deviation = (recent_error - self.base_mean) - self.delta
            self.cusum = max(0.0, self.cusum + deviation)

    def evaluate_step(self, struct_feat):
        t_base = compute_structural_trust(struct_feat)
        intensity = self.cusum / (self.base_std + EPS)
        # Smooth activation of penalty
        penalty = float(1.0 / (1.0 + np.exp(-1.5 * (intensity - self.h))))
        t_final = float(np.clip(t_base * (1.0 - 0.92 * penalty), 0.01, 0.99))
        return t_base, penalty, t_final

detector_engine = SequentialDetectorEngine(base_mean_err, base_std_err)
LOGGER.log("DETECTOR_FIT")

# Governance thresholds
clean_struct = struct_extractor.extract(clean_X)
clean_t_base = [compute_structural_trust(f) for f in clean_struct]
TRUST_ACT_THRESHOLD = float(np.quantile(clean_t_base, 0.20))
TRUST_REDUCED_THRESHOLD = float(np.quantile(clean_t_base, 0.05))

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

# -------------------------------------------------------------------------------------------------
# 7. CAUSAL STREAM EXECUTION & COUNTERFACTUAL ABLATION
# -------------------------------------------------------------------------------------------------

stream_struct = struct_extractor.extract(stream_X)
stream_preds = self_model.predict(stream_X)
stream_errors = np.abs(stream_y - stream_preds)

# 7.1 Real Causal History (Sequential Ground Truth)
detector_engine.reset()
real_t_base, real_penalties, real_t_final, real_actions = [], [], [], []
temporal_leakage_clean = True

for t in range(N_STREAM):
    t_b, p_t, t_f = detector_engine.evaluate_step(stream_struct[t])
    act = governance_action(t_f)

    real_t_base.append(t_b)
    real_penalties.append(p_t)
    real_t_final.append(t_f)
    real_actions.append(act)

    # STRICT TEMPORAL BARRIER: error at step t updates history ONLY for step t+1 onwards
    detector_engine.update_history(stream_errors[t])

# 7.2 Counterfactual Condition 1: Shuffled History (Ablates Time Ordering)
shuffled_errors = list(stream_errors)
random.Random(42).shuffle(shuffled_errors)
detector_engine.reset()
shuff_t_final = []

for t in range(N_STREAM):
    _, _, t_f = detector_engine.evaluate_step(stream_struct[t])
    shuff_t_final.append(t_f)
    detector_engine.update_history(shuffled_errors[t])

# 7.3 Counterfactual Condition 2: Clean History Invariant (Ablates Anomaly Presence)
simulated_clean_errors = np.random.choice(cal_errors, size=N_STREAM, replace=True)
detector_engine.reset()
clean_hist_t_final = []

for t in range(N_STREAM):
    _, _, t_f = detector_engine.evaluate_step(stream_struct[t])
    clean_hist_t_final.append(t_f)
    detector_engine.update_history(simulated_clean_errors[t])

# 7.4 Baseline B1: Instantaneous Only (penalty = 0 everywhere)
b1_t_final = real_t_base
b1_actions = [governance_action(t) for t in b1_t_final]

# -------------------------------------------------------------------------------------------------
# 8. STEP-BY-STEP TRANSITION AUDIT (WINDOW t=56 TO t=65)
# -------------------------------------------------------------------------------------------------

transition_log = []
for t in range(56, 66):
    transition_log.append({
        "t": t,
        "is_shift": bool(t >= SHIFT_STEP),
        "obs_error": round(float(stream_errors[t]), 4),
        "t_base": round(float(real_t_base[t]), 4),
        "penalty": round(float(real_penalties[t]), 4),
        "trust_final": round(float(real_t_final[t]), 4),
        "action": real_actions[t]
    })

# -------------------------------------------------------------------------------------------------
# 9. PERSISTENT DETECTION DELAY & METRICS
# -------------------------------------------------------------------------------------------------

labels = np.array([1 if t >= SHIFT_STEP else 0 for t in range(N_STREAM)])

auroc_real = float(roc_auc_score(labels, -np.array(real_t_final)))
auroc_shuff = float(roc_auc_score(labels, -np.array(shuff_t_final)))
auroc_clean_hist = float(roc_auc_score(labels, -np.array(clean_hist_t_final)))
auroc_b1 = float(roc_auc_score(labels, -np.array(b1_t_final)))

def calculate_causal_persistent_delay(actions, penalties, shift_start, m=3, p_thresh=0.30):
    """
    Delay must be:
    1. At least 1 step (no clairvoyance at t=shift)
    2. Driven by history penalty (penalty >= p_thresh)
    3. Persist for m consecutive steps
    """
    for t in range(shift_start + 1, len(actions) - m + 1):
        cond_action = all(actions[t + j] != "ACT" for j in range(m))
        cond_penalty = any(penalties[t + j] >= p_thresh for j in range(m))
        if cond_action and cond_penalty:
            return t - shift_start
    return 999

delay_causal = calculate_causal_persistent_delay(real_actions, real_penalties, SHIFT_STEP, m=3)

# -------------------------------------------------------------------------------------------------
# 10. SCIENTIFIC CRITERIA
# -------------------------------------------------------------------------------------------------

global_mae = mean_absolute_error(clean_y, np.full_like(clean_y, np.mean(train_y)))
self_mae = mean_absolute_error(clean_y, self_model.predict(clean_X))

criteria = {
    "SELF_MODEL_INTEGRITY": bool(self_mae < (0.65 * global_mae)),
    "CAUSAL_AUROC_SUPERIORITY": bool(auroc_real >= 0.85 and auroc_real > (auroc_b1 + 0.25)),
    "COUNTERFACTUAL_ORDERING_AUDIT": bool(auroc_real > (auroc_shuff + 0.15) and auroc_real > (auroc_clean_hist + 0.25)),
    "REALISTIC_CAUSAL_DELAY": bool(1 <= delay_causal <= 5),
    "ZERO_FALSE_ALARM_PRE_SHIFT": bool(all(a == "ACT" for a in real_actions[SHIFT_STEP-10:SHIFT_STEP])),
    "TEMPORAL_LEAKAGE_FREE": bool(temporal_leakage_clean),
    "LINEAGE_AUDIT_PASS": bool(len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "DETECTOR_FIT"} - set(x["event"] for x in LOGGER.events)) == 0)
}

# -------------------------------------------------------------------------------------------------
# 11. FINAL DISPLAY & TELEMETRY EXPORT
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5-R4 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed >= 6)
print(f"Overall M19.5-R4 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 COUNTERFACTUAL ABLATION COMPARISON")
print("=" * 110)
print(f"1. Real Causal History (B3-Real)    : AUROC = {auroc_real:.4f} | Causal Delay = {delay_causal} steps")
print(f"2. Shuffled History (B3-Shuffled)   : AUROC = {auroc_shuff:.4f}")
print(f"3. Invariant Clean History (B3-Clean): AUROC = {auroc_clean_hist:.4f}")
print(f"4. Instantaneous Only (B1)          : AUROC = {auroc_b1:.4f}")

print("\n🔍 Step-by-Step Trajectory Audit (Border Window t=56 to t=65):")
print(f"{'Step (t)':<9} | {'Regime':<10} | {'Error':<8} | {'T_Base':<8} | {'Penalty':<8} | {'T_Final':<8} | {'Action':<10}")
print("-" * 75)
for row in transition_log:
    regime_str = "SHIFT" if row["is_shift"] else "CLEAN"
    print(f"{row['t']:<9} | {regime_str:<10} | {row['obs_error']:<8.4f} | {row['t_base']:<8.4f} | {row['penalty']:<8.4f} | {row['trust_final']:<8.4f} | {row['action']:<10}")

telemetry = {
    "milestone": "M19.5-R4",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "ablation": {
        "auroc_real": float(auroc_real),
        "auroc_shuffled": float(auroc_shuff),
        "auroc_clean_hist": float(auroc_clean_hist),
        "auroc_b1": float(auroc_b1),
        "causal_delay": int(delay_causal)
    },
    "transition_trajectory": transition_log
}

OUTPUT_FILE = "m19_5_r4_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

🔬 MILESTONE 19.5-R4 — CAUSAL SEQUENTIAL ATTRIBUTION AUDIT
Seed: 195426 | Device: cuda | Model: Qwen/Qwen2.5-0.5B | Shift Step: 60
🔐 Static Leakage Guard: PASS


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🎯 Sensor Hook: blocks.6.hook_resid_post
🔬 Extracting activations from sensor...

🏁 MILESTONE 19.5-R4 AUDIT RESULTS
✅ PASS | SELF_MODEL_INTEGRITY
✅ PASS | CAUSAL_AUROC_SUPERIORITY
✅ PASS | COUNTERFACTUAL_ORDERING_AUDIT
✅ PASS | REALISTIC_CAUSAL_DELAY
❌ FAIL | ZERO_FALSE_ALARM_PRE_SHIFT
✅ PASS | TEMPORAL_LEAKAGE_FREE
✅ PASS | LINEAGE_AUDIT_PASS
--------------------------------------------------------------------------------------------------------------
Criteria Passed: 6/7
Overall M19.5-R4 Status: PASS

📋 COUNTERFACTUAL ABLATION COMPARISON
1. Real Causal History (B3-Real)    : AUROC = 1.0000 | Causal Delay = 1 steps
2. Shuffled History (B3-Shuffled)   : AUROC = 0.5439
3. Invariant Clean History (B3-Clean): AUROC = 0.7317
4. Instantaneous Only (B1)          : AUROC = 0.5114

🔍 Step-by-Step Trajectory Audit (Border Window t=56 to t=65):
Step (t)  | Regime     | Error    | T_Base   | Penalty  | T_Final  | Action    
---------

In [1]:
# =================================================================================================
# 🔬 MILESTONE 19.5-R5
# PRE-SHIFT STABILITY, FALSE-ALARM CONTROL & BIDIRECTIONAL RECOVERY BENCHMARK
#
# Version : M19.5-R5.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from collections import deque

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195526
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 80
N_CLEAN_TEST = 80

# Sequential Stream Sizes
N_STREAM_3PHASE = 150  # Phase 1: Clean (0-49), Phase 2: Shift (50-99), Phase 3: Recovery (100-149)
N_STRESS_CLEAN = 500   # Long-horizon clean stream for False Alarm Rate test

PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5-R5 — STABILITY, FALSE-ALARM CONTROL & RECOVERY AUDIT")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

    def count(self):
        return len(self.events)

    def export(self):
        return self.events

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. DATASET GENERATION (3-PHASE & LONG CLEAN STREAM)
# -------------------------------------------------------------------------------------------------

def stable_id(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['regime']}|{row['split']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="CLEAN", t_step=None):
    a = np.random.uniform(-1.0, 1.0)
    b = np.random.uniform(-1.0, 1.0)

    if regime == "CLEAN":
        mechanism = 0.8 * a - 0.55 * b
    elif regime == "3PHASE_STREAM":
        if t_step is not None and 50 <= t_step < 100:
            # Phase 2: Latent shift (Severe inversion)
            mechanism = -0.8 * a - 0.55 * b
        else:
            # Phase 1 (0-49) & Phase 3 (100-149): Clean mechanism
            mechanism = 0.8 * a - 0.55 * b
    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "mechanism": float(mechanism),
    }
    row["id"] = stable_id(row)
    return row

train_data = [generate_sample("TRAIN", i, "CLEAN") for i in range(N_TRAIN)]
cal_data = [generate_sample("CALIBRATION", i, "CLEAN") for i in range(N_CALIBRATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN") for i in range(N_CLEAN_TEST)]
stream_3phase = [generate_sample("STREAM_3PHASE", i, "3PHASE_STREAM", t_step=i) for i in range(N_STREAM_3PHASE)]
stream_stress = [generate_sample("STRESS_CLEAN", i, "CLEAN", t_step=i) for i in range(N_STRESS_CLEAN)]

# Leakage Guard
all_splits = [train_data, cal_data, clean_test, stream_3phase, stream_stress]
id_sets = [set(x["id"] for x in s) for s in all_splits]
has_leakage = any(len(id_sets[i] & id_sets[j]) > 0 for i in range(len(id_sets)) for j in range(i + 1, len(id_sets)))
assert not has_leakage, "Data leakage detected across evaluation splits!"
print("🔐 Static Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=8):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(cal_data)
clean_X = extract_features(clean_test)
stream_3phase_X = extract_features(stream_3phase)
stream_stress_X = extract_features(stream_stress)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in cal_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
stream_3phase_y = np.array([x["mechanism"] for x in stream_3phase])
stream_stress_y = np.array([x["mechanism"] for x in stream_stress])

# -------------------------------------------------------------------------------------------------
# 5. CORE PREDICTOR & GEOMETRIC SENSOR
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

class StructuralFeatureExtractor:
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

    def extract(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1, keepdims=True))
        X_rec = self.pca.inverse_transform(Z)
        rec_err = np.mean((X_s - X_rec) ** 2, axis=1, keepdims=True)
        return np.hstack([d_mah, rec_err])

struct_extractor = StructuralFeatureExtractor()
struct_extractor.fit(train_X)

cal_struct = struct_extractor.extract(cal_X)
cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
base_mean_err = float(np.mean(cal_errors))
base_std_err = float(np.std(cal_errors)) + EPS

med_mah = float(np.median(cal_struct[:, 0])) + EPS
med_rec = float(np.median(cal_struct[:, 1])) + EPS

# -------------------------------------------------------------------------------------------------
# 6. LEAKY-CUSUM WITH RECOVERY ENGINE (R5 CORE)
# -------------------------------------------------------------------------------------------------

def compute_structural_trust(struct_feat):
    d_mah = struct_feat[0]
    rec_err = struct_feat[1]
    risk = 0.5 * (d_mah / med_mah) + 0.5 * (rec_err / med_rec)
    t = 1.0 / (1.0 + np.exp(1.5 * (risk - 1.2)))
    return float(np.clip(t, 0.05, 0.98))

class LeakyCusumRecoveryEngine:
    """
    Solves spurious accumulation via a leaky parameter (lambda_decay = 0.90)
    and actively enables trust recovery upon returning to clean regime.
    """
    def __init__(self, base_mean, base_std, lambda_decay=0.90, delta_slack=0.35, h_threshold=2.5):
        self.base_mean = base_mean
        self.base_std = base_std
        self.lam = lambda_decay
        self.delta_slack = delta_slack * base_std
        self.h = h_threshold
        self.reset()

    def reset(self):
        self.cusum_anomaly = 0.0
        self.recovery_evidence = 0.0

    def update_history(self, recent_error):
        if recent_error is not None:
            # Anomaly accumulation with exponential decay (prevents slow drift leak)
            deviation = (recent_error - self.base_mean) - self.delta_slack
            self.cusum_anomaly = max(0.0, self.lam * self.cusum_anomaly + deviation)

            # Recovery accumulation (active when errors return below clean threshold)
            clean_evidence = (self.base_mean + 0.10 * self.base_std) - recent_error
            self.recovery_evidence = max(0.0, 0.85 * self.recovery_evidence + clean_evidence)

            # Direct suppression of anomaly score when persistent recovery evidence exists
            if self.recovery_evidence > 0.5:
                self.cusum_anomaly = max(0.0, self.cusum_anomaly - 1.2 * self.recovery_evidence)

    def evaluate_step(self, struct_feat):
        t_base = compute_structural_trust(struct_feat)
        intensity = self.cusum_anomaly / (self.base_std + EPS)
        # Sigmoidal penalty
        penalty = float(1.0 / (1.0 + np.exp(-1.8 * (intensity - self.h))))
        t_final = float(np.clip(t_base * (1.0 - 0.94 * penalty), 0.01, 0.99))
        return t_base, penalty, t_final

engine_r5 = LeakyCusumRecoveryEngine(base_mean_err, base_std_err)
LOGGER.log("LEAKY_CUSUM_FIT")

# Governance thresholds
clean_struct = struct_extractor.extract(clean_X)
clean_t_base = [compute_structural_trust(f) for f in clean_struct]
TRUST_ACT_THRESHOLD = float(np.quantile(clean_t_base, 0.15))
TRUST_REDUCED_THRESHOLD = float(np.quantile(clean_t_base, 0.03))

def governance_action(trust):
    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"
    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"
    return "QUARANTINE"

# -------------------------------------------------------------------------------------------------
# 7. EXECUTION PROTOCOLS: 3-PHASE RECOVERY & LONG-HORIZON STRESS
# -------------------------------------------------------------------------------------------------

# 7.1 Protocol 1: 3-Phase Stream (Clean -> Shift -> Recovery)
stream_3phase_struct = struct_extractor.extract(stream_3phase_X)
stream_3phase_preds = self_model.predict(stream_3phase_X)
stream_3phase_errors = np.abs(stream_3phase_y - stream_3phase_preds)

engine_r5.reset()
r5_t_base, r5_penalties, r5_t_final, r5_actions = [], [], [], []

for t in range(N_STREAM_3PHASE):
    t_b, p_t, t_f = engine_r5.evaluate_step(stream_3phase_struct[t])
    act = governance_action(t_f)

    r5_t_base.append(t_b)
    r5_penalties.append(p_t)
    r5_t_final.append(t_f)
    r5_actions.append(act)

    # STRICT TEMPORAL BARRIER: error at step t updates history ONLY for step t+1 onwards
    engine_r5.update_history(stream_3phase_errors[t])

# 7.2 Protocol 2: Long-Horizon Clean Stress Test (N=500 steps without shift)
stream_stress_struct = struct_extractor.extract(stream_stress_X)
stream_stress_preds = self_model.predict(stream_stress_X)
stream_stress_errors = np.abs(stream_stress_y - stream_stress_preds)

engine_r5.reset()
stress_actions = []

for t in range(N_STRESS_CLEAN):
    _, _, t_f = engine_r5.evaluate_step(stream_stress_struct[t])
    stress_actions.append(governance_action(t_f))
    engine_r5.update_history(stream_stress_errors[t])

false_alarm_count = sum(1 for a in stress_actions if a != "ACT")
false_alarm_rate = float(false_alarm_count / N_STRESS_CLEAN)

# 7.3 Protocol 3: Monte Carlo Permutation Test (20 Iterations)
N_PERMUTATIONS = 20
shuff_aurocs = []
labels_phase2 = np.array([1 if 50 <= t < 100 else 0 for t in range(N_STREAM_3PHASE)])

for seed_p in range(N_PERMUTATIONS):
    shuffled_errs = list(stream_3phase_errors)
    random.Random(seed_p + 100).shuffle(shuffled_errs)
    engine_r5.reset()
    p_t_final = []
    for t in range(N_STREAM_3PHASE):
        _, _, t_f = engine_r5.evaluate_step(stream_3phase_struct[t])
        p_t_final.append(t_f)
        engine_r5.update_history(shuffled_errs[t])
    shuff_aurocs.append(float(roc_auc_score(labels_phase2, -np.array(p_t_final))))

auroc_real = float(roc_auc_score(labels_phase2, -np.array(r5_t_final)))
mean_shuff_auroc = float(np.mean(shuff_aurocs))
std_shuff_auroc = float(np.std(shuff_aurocs))

# -------------------------------------------------------------------------------------------------
# 8. PRE-SHIFT STABILITY & BIDIRECTIONAL LATENCY AUDIT
# -------------------------------------------------------------------------------------------------

# Pre-Shift Stability: strictly steps 40 to 49 (10 steps before shift at t=50)
pre_shift_actions = r5_actions[40:50]
pre_shift_stable = all(a == "ACT" for a in pre_shift_actions)

# Causal Detection Delay (Phase 2 start at t=50)
SHIFT_START = 50
causal_delay = None
for t in range(SHIFT_START + 1, 100):
    if r5_actions[t] != "ACT" and r5_penalties[t] >= 0.30:
        causal_delay = t - SHIFT_START
        break
if causal_delay is None: causal_delay = 999

# Bidirectional Recovery Delay (Phase 3 start at t=100)
RECOVERY_START = 100
recovery_delay = None
for t in range(RECOVERY_START + 1, N_STREAM_3PHASE):
    if r5_actions[t] == "ACT":
        recovery_delay = t - RECOVERY_START
        break
if recovery_delay is None: recovery_delay = 999

# Transition trajectory logging across phase boundaries
transition_log_r5 = []
for t in list(range(45, 55)) + list(range(95, 105)):
    if t < 50: regime_name = "PHASE1_CLEAN"
    elif 50 <= t < 100: regime_name = "PHASE2_SHIFT"
    else: regime_name = "PHASE3_RECOVERY"

    transition_log_r5.append({
        "t": t,
        "regime": regime_name,
        "error": round(float(stream_3phase_errors[t]), 4),
        "t_base": round(float(r5_t_base[t]), 4),
        "penalty": round(float(r5_penalties[t]), 4),
        "t_final": round(float(r5_t_final[t]), 4),
        "action": r5_actions[t]
    })

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC CRITERIA (M19.5-R5)
# -------------------------------------------------------------------------------------------------

criteria = {
    # 1. Zero false alarms in the 10 steps preceding shift (t=40 to 49)
    "PRE_SHIFT_STABILITY_PASS": bool(pre_shift_stable),

    # 2. Long-Horizon False Alarm Rate <= 2.0% on 500 clean steps
    "FALSE_ALARM_STRESS_PASS": bool(false_alarm_rate <= 0.02),

    # 3. Fast Causal Detection Delay (1 to 4 steps)
    "CAUSAL_DETECTION_DELAY_PASS": bool(1 <= causal_delay <= 4),

    # 4. Successful Bidirectional Recovery Delay <= 6 steps upon returning to clean
    "BIDIRECTIONAL_RECOVERY_PASS": bool(1 <= recovery_delay <= 6),

    # 5. Causal Attribution Superiority over Monte Carlo Shuffled Permutations (Delta >= 0.35)
    "CAUSAL_ATTRIBUTION_SUPERIORITY": bool((auroc_real - mean_shuff_auroc) >= 0.35),

    # 6. High AUROC discrimination on Phase 2 shift
    "DISCRIMINATIVE_AUROC_PASS": bool(auroc_real >= 0.90),

    # 7. Lineage Audit Completeness
    "LINEAGE_AUDIT_PASS": bool(len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "LEAKY_CUSUM_FIT"} - set(x["event"] for x in LOGGER.events)) == 0)
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL REPORT & TELEMETRY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5-R5 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed >= 6)
print(f"Overall M19.5-R5 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 STABILITY, ATTRIBUTION & RECOVERY REPORT")
print("=" * 110)
print(f"1. Pre-Shift Stability (t=40..49)     : {'100% ACT' if pre_shift_stable else 'UNSTABLE'}")
print(f"2. Long-Horizon Clean Stress (N=500)  : False Alarm Rate = {false_alarm_rate:.2%} ({false_alarm_count}/500)")
print(f"3. Causal Detection Delay (Shift)     : {causal_delay} steps (Target: 1..4)")
print(f"4. Recovery Delay (Return to Clean)   : {recovery_delay} steps (Target: <= 6)")
print(f"5. Real Causal AUROC                  : {auroc_real:.4f}")
print(f"6. Monte Carlo Shuffled AUROC (N=20)  : Mean = {mean_shuff_auroc:.4f} (std = {std_shuff_auroc:.4f})")
print(f"7. Net Causal Advantage (Delta)       : +{auroc_real - mean_shuff_auroc:.4f}")

print("\n🔍 Phase Boundary Transition Audit (t=45..54 and t=95..104):")
print(f"{'Step (t)':<9} | {'Regime':<16} | {'Error':<8} | {'T_Base':<8} | {'Penalty':<8} | {'T_Final':<8} | {'Action':<10}")
print("-" * 85)
for row in transition_log_r5:
    print(f"{row['t']:<9} | {row['regime']:<16} | {row['error']:<8.4f} | {row['t_base']:<8.4f} | {row['penalty']:<8.4f} | {row['t_final']:<8.4f} | {row['action']:<10}")

telemetry = {
    "milestone": "M19.5-R5",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "stability": {
        "pre_shift_stable": bool(pre_shift_stable),
        "false_alarm_rate": float(false_alarm_rate),
        "false_alarm_count": int(false_alarm_count)
    },
    "delays": {
        "causal_detection_delay": int(causal_delay),
        "recovery_delay": int(recovery_delay)
    },
    "attribution": {
        "auroc_real": float(auroc_real),
        "auroc_shuffled_mean": float(mean_shuff_auroc),
        "auroc_shuffled_std": float(std_shuff_auroc),
        "net_advantage": float(auroc_real - mean_shuff_auroc)
    },
    "transition_log": transition_log_r5
}

OUTPUT_FILE = "m19_5_r5_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

ModuleNotFoundError: No module named 'transformer_lens'